[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imageomics/sage-summer-2026-bioclip/blob/main/notebooks/peromyscus.ipynb)

# Compare BioCLIP model generations on a fine-grained species task

This notebook uses fine-grained classification of *Peromyscus* species as a running scientific task. We compare three models:

1. [BioCLIP](https://huggingface.co/imageomics/bioclip)
2. [BioCLIP 2](https://huggingface.co/imageomics/bioclip-2)
3. [BioCLIP 2.5](https://huggingface.co/imageomics/bioclip-2.5-vith14)

An image encoder maps pixels to a fixed-length vector called an **embedding**. Images with similar model representations tend to point in similar directions in the embedding space. A **frozen** encoder is evaluated without changing its learned weights.

For each model, we test two ways to turn embeddings into species predictions:

- **Zero-shot classification** compares an image embedding with text embeddings for the candidate species. It does not fit a classifier from labeled images.
- **Few-shot classification** fits a small classifier on frozen image embeddings using a limited number of labeled examples.

Every model comparison uses the same images, labels, train/test split, and few-shot support sets. We then adapt the final visual block of BioCLIP 2.5 using a validation subset of its labeled training images. The later sections compare the original and adapted representations with dynamic INT8 post-training quantization.

## Notebook flow

1. **Load and inspect the data.** Read *Peromyscus* metadata, images, and stored embeddings from Lance.
2. **Define the task.** Select ten species with 50 images each, then make one shared 40/10 train/test split per species.
3. **Inspect the embedding spaces.** Normalize the three models' image embeddings and visualize their species neighborhoods.
4. **Run zero-shot classification.** Compare images with taxonomic text prompts and training-aligned prompt prototypes.
5. **Train classifiers on frozen embeddings.** Fit linear SVMs, inspect errors, and vary the shot count from 1 to 40 labeled images per species.
6. **Adapt the image encoder.** Train the final BioCLIP 2.5 visual block toward fixed taxonomic text prototypes and evaluate the adapted embeddings.
7. **Quantize the image encoders.** Apply dynamic W8A8 PTQ and compare storage, measured inference time, and embedding agreement.
8. **Repeat the scientific task.** Rerun prototype and SVM evaluation with W8A8 embeddings and inspect changed predictions.
9. **Examine genus-only camera-trap images.** Interactively compare predictions from the original and adapted models without treating them as ground truth.
10. **Check a broader set of tasks.** Use the 164-task NeWT portion of BioBench to compare BioCLIP 2.5 before and after quantization.

The images, metadata, and FP32 embeddings come from release [`v0.1.0` of `imageomics/fine-grained-challenges`](https://huggingface.co/datasets/imageomics/fine-grained-challenges/tree/v0.1.0).

> The labeled task images are marked as present in the BioCLIP 2 and BioCLIP 2.5 training corpora. The train/test split separates task adaptation and evaluation images, but it does not make the evaluation images novel to the foundation models. Treat this as an adaptation exercise rather than an external test of generalization.


## Environment

A compatible CUDA GPU is used when available. The notebook checks that the installed PyTorch build can execute on the assigned GPU and falls back to CPU if it cannot.

The first part reads image embeddings from Lance and only encodes taxonomic text prompts. The fine-tuning, quantization, and BioBench sections perform image inference and benefit substantially from CUDA.

For a local Jupyter environment, run these commands from the repository root and select the resulting kernel:

```bash
uv venv .venv-peromyscus --python 3.12
uv pip install --python .venv-peromyscus/bin/python -r notebooks/requirements.txt
.venv-peromyscus/bin/python -m ipykernel install --user --name sage-peromyscus --display-name "Sage Peromyscus"
```

The Lance version below supports a shared session cache and coalesced reads for remote `hf://` datasets.

On a fresh NDP server, run the environment cell and then refresh the browser tab once before continuing. This lets JupyterLab load the newly installed widget frontend. If an interactive output appears as `HBox(children=...)`, refresh the tab and rerun that output cell; the kernel and completed cells remain available.

In [ ]:
# Colab has a disposable environment, so uv installs into the active kernel.
%pip install -q uv
!uv pip install --system -q "pylance>=8,<9" "pyarrow>=20,<26" "polars>=1.30,<2" "open_clip_torch>=3,<4" "torch>=2.11" "torchvision>=0.26" "scikit-learn>=1.6,<2" "matplotlib>=3.9,<4" "torchao>=0.17,<0.18" "requests>=2.32,<3" "tqdm>=4.66,<5" "ipywidgets>=8,<9" "jupyterlab_widgets>=3,<4" "widgetsnbextension>=4,<5" "plotly>=6,<7" "anywidget>=0.9,<1" pillow

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

In [ ]:
# Locate the notebook helpers before importing them in the setup cell below.
import sys
from pathlib import Path
from urllib.request import urlretrieve

sidecar_names = [
    "taxonomic_prompts.py",
    "embedding_bundles.py",
    "fine_tuning_helpers.py",
    "interactive_camera_trap.py",
    "interactive_embedding_plot.py",
    "quantization_helpers.py",
    "biobench_helpers.py",
]
sidecar_directory = next((
    directory for directory in [
        Path("."), Path("notebooks")
    ]
    if all((directory / name).exists() for name in sidecar_names)
), None)
if sidecar_directory is None:
    sidecar_directory = Path(".notebook_helpers")
    sidecar_directory.mkdir(exist_ok=True)
    for sidecar_name in sidecar_names:
        urlretrieve(
            "https://raw.githubusercontent.com/Imageomics/"
            "sage-summer-2026-bioclip/main/notebooks/"
            f"{sidecar_name}",
            sidecar_directory / sidecar_name,
        )
sys.path.insert(0, str(sidecar_directory.resolve()))

## Optional: authenticate with Hugging Face

Authentication is unnecessary when the dataset repository and models are public. If access is restricted or Hub requests are throttled in a shared environment, uncomment the login lines below and use a personal read token. Do not place a token directly in notebook source.

In [ ]:
# Keep authentication optional; enable it when access or rate limits require it.
import os

# from huggingface_hub import get_token, login

# login()
# os.environ["HF_TOKEN"] = get_token()

In [ ]:
# Establish the data source, model revisions, and shared analysis settings.
import gc
import io

import lance
import matplotlib.pyplot as plt
import numpy as np
import open_clip
import polars as pl
import requests
import torch
from huggingface_hub import snapshot_download
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.svm import SVC

from embedding_bundles import (
    EmbeddingBundle,
    evaluate_few_shot_bundles,
    ids_digest,
    producer_manifest,
)
from interactive_embedding_plot import EmbeddingScatterViewer
from interactive_camera_trap import (
    ClassificationErrorViewer,
    ImageBrowser,
    format_details,
    format_table,
    load_megadetector_boxes,
)
from quantization_helpers import select_device

# Keep the labeled task separate from the unseen camera-trap cohort.
REPO_ID = "imageomics/fine-grained-challenges"
DATASET_REVISION = "v0.1.0"
CHALLENGE_GROUP = "Peromyscus"
LANCE_URI = f"hf://datasets/{REPO_ID}/data/challenges.lance"
LANCE_STORAGE_OPTIONS = {"revision": DATASET_REVISION}
LABELED_TASK_FILTER = (
    f"challenge_group = '{CHALLENGE_GROUP}' "
    "AND species IS NOT NULL "
    "AND source_dataset != 'lila-bc'"
)
LILA_TRIAL_FILTER = (
    f"challenge_group = '{CHALLENGE_GROUP}' "
    "AND source_dataset = 'lila-bc'"
)
MIN_IMAGES_PER_SPECIES = 50
MAX_SPECIES = 10
IMAGES_PER_SPECIES = 50
PREVIEW_SPECIES = 8
RANDOM_SEED = 42
TEST_FRACTION = 0.20
METADATA_CACHE_BYTES = 256 * 1024**2
DEVICE = select_device()

# Pin the checkpoint and stored embedding column for each model generation.
MODEL_SPECS = {
    "BioCLIP": {
        "repo_id": "imageomics/bioclip",
        "revision": "ce901ab3c6a913f9e9ef94ce6d27761069f4f01c",
        "column": "emb_bioclip",
        "dimension": 512,
        "weight_file": "open_clip_pytorch_model.bin",
    },
    "BioCLIP 2": {
        "repo_id": "imageomics/bioclip-2",
        "revision": "2957b322090f9cb17ae72c71981c7218a28d81e0",
        "column": "emb_bioclip2",
        "dimension": 768,
        "weight_file": "open_clip_model.safetensors",
    },
    "BioCLIP 2.5": {
        "repo_id": "imageomics/bioclip-2.5-vith14",
        "revision": "191d741545e4c741cdef4b22c6eb69c945c1e592",
        "column": "emb_bioclip2p5",
        "dimension": 1024,
        "weight_file": "open_clip_model.safetensors",
    },
}
EMBEDDING_COLUMNS = [spec["column"] for spec in MODEL_SPECS.values()]

# Clear large state left by a previous execution before opening Lance.
for stale_name in [
    "model",
    "raw_embeddings",
    "embedding_bundles",
    "quantized_embedding_bundles",
    "lila_embedding_bundles",
    "quantized_lila_embedding_bundles",
    "adapted_embedding_bundle",
    "adapted_quantized_embedding_bundle",
    "adapted_lila_embedding_bundle",
    "adapted_quantized_lila_embedding_bundle",
    "sample_table",
    "image_bytes",
    "fig",
]:
    globals().pop(stale_name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

lance_session = lance.Session(metadata_cache_size_bytes=METADATA_CACHE_BYTES)
lance_ds = lance.dataset(
    LANCE_URI,
    session=lance_session,
    storage_options=LANCE_STORAGE_OPTIONS,
)

if DEVICE.type == "cuda":
    print(f"Device: {torch.cuda.get_device_name(DEVICE)}")
else:
    print("Device: CPU")
print(f"Dataset: {REPO_ID}@{DATASET_REVISION}")
print("Models:", ", ".join(MODEL_SPECS))


# 1/10 Load and inspect the data.

## Preview the dataset

Lance first projects the scalar metadata for the configured challenge group. One `LanceDataset` and its in-memory session cache are reused for every query in this notebook. The cache holds manifests and file metadata, not the image column or a copy of the dataset.

Images are binary objects in the table. One bounded query retrieves representative images for the preview without downloading the complete image column. Each image retains its own license.

In [ ]:
# Read lightweight metadata before retrieving image bytes or embeddings.
metadata_columns = [
    "uuid",
    "challenge_group",
    "seen_in_training",
    "species",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "common_name",
    "source_dataset",
    "source_id",
    "license_name",
]
group_metadata = pl.from_arrow(
    lance_ds.scanner(
        filter=LABELED_TASK_FILTER,
        columns=metadata_columns,
    ).to_table()
)
# Select one record per species for an initial image inspection.
preview_metadata = (
    group_metadata.drop_nulls("species")
    .sort(["species", "uuid"])
    .unique(subset=["species"], maintain_order=True)
    .head(PREVIEW_SPECIES)
)
display(preview_metadata.drop("uuid"))
print(f"Lance session cache: {lance_session.size_bytes() / 1024**2:.2f} MiB")

def quote_lance_string(value):
    return "'" + value.replace("'", "''") + "'"

# Fetch only the selected preview images and connect them to a browser.
preview_uuid_values = ", ".join(
    quote_lance_string(value) for value in preview_metadata["uuid"]
)
preview_rows = lance_ds.scanner(
    filter=f"uuid IN ({preview_uuid_values})",
    columns=["uuid", "species", "common_name", "license_name", "image"],
).to_table().sort_by([("uuid", "ascending")]).to_pylist()

preview_browser = ImageBrowser(
    images=[row["image"] for row in preview_rows],
    details=[
        format_details([
            ("Species", row["species"]),
            ("Common name", row["common_name"] or "not recorded"),
            ("Image license", row["license_name"]),
        ])
        for row in preview_rows
    ],
    title=f"{CHALLENGE_GROUP} images in the dataset",
    random_seed=RANDOM_SEED,
)
preview_browser.display()
del preview_rows
gc.collect()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

# 2/10 Define the task.

## Define the classification task

A benchmark needs a candidate label set and an evaluation population. Here, the labels are *Peromyscus* species with enough images for the shared task and adaptation splits.

Each retained species supplies:

- 40 labeled training images and 10 test images
- learning curves through 40 labeled training images
- within the adaptation section, 30 optimization images and 10 validation images drawn from the 40-image training set

Rows identified only to genus are excluded because they do not supply a species label. Caps on species and images keep the classes balanced and the remote Lance query bounded.


In [ ]:
# Select ten sufficiently represented species and balance their image counts.
class_counts = (
    group_metadata.drop_nulls("species")
    .group_by("species")
    .len(name="images")
    .sort(["images", "species"], descending=[True, False])
)
eligible_class_counts = class_counts.filter(
    pl.col("images") >= MIN_IMAGES_PER_SPECIES
)
selected_species = (
    eligible_class_counts.head(MAX_SPECIES)["species"].sort().to_list()
)
# Use the same number of images per species so aggregate metrics are not
# dominated by the most common class.
selected_metadata = (
    group_metadata.filter(pl.col("species").is_in(selected_species))
    .sort(["species", "uuid"])
    .group_by("species", maintain_order=True)
    .head(IMAGES_PER_SPECIES)
)
selected_uuids = selected_metadata["uuid"].sort().to_list()

print(f"Rows in labeled task pool: {group_metadata.height:,}")
print(f"Species represented: {class_counts.height}")
print(f"Species retained: {len(selected_species)}")
print(f"Images retained: {len(selected_uuids):,}")
class_counts.filter(pl.col("species").is_in(selected_species))


## Fetch the selected images and embeddings

Lance applies both the row filter and the column projection remotely. The query retrieves only the selected UUIDs, image bytes, metadata, and three embedding columns. The embeddings were computed from these stored image bytes with the pinned model revisions listed above.

Keeping the images and embeddings in one table lets us compare stored FP32 representations with fresh quantized inference on exactly the same inputs.

In [ ]:
# Retrieve the selected images and three stored representations in one query.
uuid_values = ", ".join(quote_lance_string(value) for value in selected_uuids)
sample_filter = (
    f"challenge_group = '{CHALLENGE_GROUP}' "
    f"AND uuid IN ({uuid_values})"
)
sample_columns = [
    "uuid",
    "species",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "common_name",
    "source_dataset",
    "source_id",
    "seen_in_training",
    "license_name",
    *EMBEDDING_COLUMNS,
    "image",
]
# Fetch the selected columns together. Coalescing this read avoids many small
# Hub API requests and keeps each UUID aligned with its image and embeddings.
sample_table = lance_ds.scanner(
    filter=sample_filter,
    columns=sample_columns,
).to_table().sort_by([("uuid", "ascending")])

# Preserve row order while separating metadata, image bytes, and embeddings.
sample_metadata = pl.from_arrow(sample_table.drop([*EMBEDDING_COLUMNS, "image"]))
image_bytes = sample_table["image"].combine_chunks().to_pylist()
raw_embeddings = {}
for model_name, spec in MODEL_SPECS.items():
    raw_embeddings[model_name] = np.asarray(
        sample_table[spec["column"]].combine_chunks().values,
        dtype=np.float32,
    ).reshape(-1, spec["dimension"])

uuids = np.asarray(sample_metadata["uuid"].to_list(), dtype=np.str_)
labels = sample_metadata["species"].to_numpy()
del sample_table
gc.collect()

# Check the retrieved shapes, training exposure, and image licenses.
print(f"Selected images: {len(labels):,}")
for model_name, features in raw_embeddings.items():
    print(f"{model_name}: {features.shape}")
print(f"Lance session cache: {lance_session.size_bytes() / 1024**2:.2f} MiB")
print("Foundation-model training exposure:")
display(sample_metadata.select("seen_in_training").unique())
print("Image licenses in this subset:")
sample_metadata["license_name"].value_counts().sort("count", descending=True)

In [ ]:
# Browse the balanced task images before using their labels for evaluation.
def decode_image(value):
    with Image.open(io.BytesIO(value)) as image:
        return image.convert("RGB")

task_browser = ImageBrowser(
    images=image_bytes,
    details=[
        format_details([
            ("Species", row["species"]),
            ("Common name", row["common_name"] or "not recorded"),
            ("Image license", row["license_name"]),
        ])
        for row in sample_metadata.to_dicts()
    ],
    title=f"{CHALLENGE_GROUP} task images",
    random_seed=RANDOM_SEED,
)
task_browser.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

# 3/10 Inspect the embedding spaces.

## Prepare the embedding spaces

The Lance columns contain raw encoder outputs. Cosine similarity compares vector direction, so each vector is divided by its Euclidean norm to produce a unit-length copy. The raw arrays remain unchanged.

The norm summary is also a data check: zero or non-finite norms would make cosine scoring invalid.

In [ ]:
# Normalize each stored representation and attach its producer metadata.
embedding_bundles = {}
embedding_rows = []
for model_name, values in raw_embeddings.items():
    norms = np.linalg.norm(values, axis=1)
    spec = MODEL_SPECS[model_name]
    embedding_bundles[model_name] = EmbeddingBundle.create(
        uuids,
        values,
        producer_manifest(
            bundle_id=f"{CHALLENGE_GROUP}:{model_name}:fp32",
            model_name=model_name,
            repo_id=spec["repo_id"],
            revision=spec["revision"],
            precision="FP32",
            evidence_type="dataset-provided embedding",
            framework="OpenCLIP",
            framework_version=open_clip.__version__,
            preprocessing="OpenCLIP evaluation transform from pinned model config",
            quantization=None,
            dataset={
                "repo_id": REPO_ID,
                "subset": CHALLENGE_GROUP,
            },
        ),
    )
    embedding_rows.append({
        "model": model_name,
        "bundle_id": embedding_bundles[model_name].manifest["bundle_id"],
        "dimension": values.shape[1],
        "minimum_raw_norm": float(norms.min()),
        "mean_raw_norm": float(norms.mean()),
        "maximum_raw_norm": float(norms.max()),
    })

pl.DataFrame(embedding_rows)

## Species structure in the embedding spaces

t-SNE places nearby high-dimensional embeddings near one another in two dimensions. It preserves local neighborhoods rather than global distance, so the useful signal is where species form groups or overlap, not how far apart two groups appear. Each panel is fit independently; rotation, scale, and positions are not comparable between models.

The embeddings are unit-normalized, making Euclidean and cosine distance equivalent for neighbor ordering. The fit otherwise uses scikit-learn's defaults, with a fixed random seed so the view is reproducible. Select a marker to inspect its source image below that panel.

In [ ]:
# Fit one t-SNE per model and link every point to its source image.
species_palette_names = sorted(np.unique(labels).tolist())
palette = plt.get_cmap("tab10")
SPECIES_COLORS = {
    species: palette(index)
    for index, species in enumerate(species_palette_names)
}
tsne_coordinates = {
    model_name: TSNE(
        metric="euclidean",
        random_state=RANDOM_SEED,
    ).fit_transform(bundle.features)
    for model_name, bundle in embedding_bundles.items()
}
embedding_space_viewer = EmbeddingScatterViewer(
    coordinates_by_panel=tsne_coordinates,
    images=image_bytes,
    labels=labels,
    uuids=uuids,
    colors=SPECIES_COLORS,
    title=f"{CHALLENGE_GROUP} image embeddings by recorded species",
)
embedding_space_viewer.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

# 4/10 Run zero-shot classification.

## Training-aligned text prototypes

A zero-shot classifier needs one text vector, or **prototype**, for each candidate species. We compare two definitions:

1. **Plain taxonomic lineage:** encode the available lineage terms from kingdom through the most specific rank, without rank labels or prompt wording. This is called `taxonomic_name` in the training data.
2. **Training-template prototype:** encode the ten text formats used during BioCLIP training, average their unit vectors, and normalize the result.

For `sci`, the most specific Linnaean term is used, including the full binomial at species rank. If a common name is absent, the same Linnaean term fills the common-name field, matching the training-data fallback. Duplicate prompts are retained because they came from distinct training template types.

The predicted species is the prototype with the highest cosine similarity to the image. No labeled images are used to fit these classifiers.

The confusion matrices are normalized by true species. Each row contains ten test images, so `n=10` is the row support and one image represents 0.10 of that row.

In [ ]:
# Build two candidate-text conditions for every species.
from taxonomic_prompts import (
    TRAINING_TEMPLATE_NAMES,
    build_class_definitions,
    build_training_prompt_ensemble,
    training_template_prompts,
)

RAW_METHOD = "Plain taxonomic lineage"
PROTOTYPE_METHOD = "Training-template prototype"
ZERO_SHOT_METHODS = [RAW_METHOD, PROTOTYPE_METHOD]

class_definitions = build_class_definitions(
    sample_metadata, group_metadata
)

class_names = [definition["class_name"] for definition in class_definitions]
prompt_preview_definition = class_definitions[0]
prompt_preview_common_name = prompt_preview_definition["common_names"][0]
# Preview the ten training formats for one species.
display(pl.DataFrame({
    "template": TRAINING_TEMPLATE_NAMES,
    "prompt": training_template_prompts(
        prompt_preview_definition, prompt_preview_common_name
    ),
}))
# Retain both the plain lineage and the complete training-template set.
raw_prompts = [
    definition["taxonomic_name"] for definition in class_definitions
]
ensemble_prompts, ensemble_slices = build_training_prompt_ensemble(
    class_definitions
)

In [ ]:
# Fix one held-out split, then test both text conditions for each model.
indices = np.arange(len(labels))
# Create one stratified split before evaluating any model. This prevents a
# favorable split for one representation from changing the comparison.
train_indices, test_indices = train_test_split(
    indices,
    test_size=TEST_FRACTION,
    random_state=RANDOM_SEED,
    stratify=labels,
)
y_test = labels[test_indices]
evaluation_split = {
    "random_seed": RANDOM_SEED,
    "test_fraction": TEST_FRACTION,
    "train_ids_sha256": ids_digest(uuids[train_indices]),
    "test_ids_sha256": ids_digest(uuids[test_indices]),
    "train_images": len(train_indices),
    "test_images": len(test_indices),
}

# Allocate results shared by the metric table and later image inspection.
zero_shot_predictions = {method: {} for method in ZERO_SHOT_METHODS}
zero_shot_probabilities = {method: {} for method in ZERO_SHOT_METHODS}
zero_shot_prototypes = {method: {} for method in ZERO_SHOT_METHODS}
zero_shot_metric_rows = []
tokenizer = open_clip.get_tokenizer("ViT-B-16")

# Load one model at a time and encode its candidate text.
for model_name, spec in MODEL_SPECS.items():
    print(f"Loading {model_name}...")
    snapshot = snapshot_download(
        repo_id=spec["repo_id"],
        revision=spec["revision"],
        allow_patterns=["open_clip_config.json", spec["weight_file"]],
    )
    model, _, _ = open_clip.create_model_and_transforms(
        f"local-dir:{snapshot}", device=DEVICE, precision="fp32"
    )
    model.eval()

    # Encode the single plain-lineage prompt for each species.
    raw_tokens = tokenizer(raw_prompts).to(DEVICE)
    with torch.inference_mode():
        raw_prototypes = model.encode_text(raw_tokens, normalize=True)
    raw_prototypes = raw_prototypes.float().cpu().numpy()

    # Average the ten unit text embeddings into one prototype per species.
    ensemble_tokens = tokenizer(ensemble_prompts).to(DEVICE)
    with torch.inference_mode():
        ensemble_text_features = model.encode_text(
            ensemble_tokens, normalize=True
        )
    ensemble_text_features = ensemble_text_features.float().cpu().numpy()
    ensemble_prototypes = np.stack([
        ensemble_text_features[prompt_slice].mean(axis=0)
        for prompt_slice in ensemble_slices
    ])
    ensemble_prototypes /= np.linalg.norm(
        ensemble_prototypes, axis=1, keepdims=True
    )

    # Ask which candidate prototype is closest to each held-out image.
    prototypes_by_method = {
        RAW_METHOD: raw_prototypes,
        PROTOTYPE_METHOD: ensemble_prototypes,
    }
    temperature = float(model.logit_scale.exp().detach().cpu())
    for method, prototypes in prototypes_by_method.items():
        zero_shot_prototypes[method][model_name] = prototypes.copy()
        similarities = (
            embedding_bundles[model_name].features[test_indices]
            @ prototypes.T
        )
        predictions = np.asarray(class_names)[similarities.argmax(axis=1)]
        probabilities = torch.softmax(
            torch.from_numpy(similarities * temperature), dim=1
        ).numpy()
        zero_shot_predictions[method][model_name] = predictions
        zero_shot_probabilities[method][model_name] = probabilities
        zero_shot_metric_rows.append({
            "model": model_name,
            "method": method,
            "accuracy": accuracy_score(y_test, predictions),
            "macro_f1": f1_score(y_test, predictions, average="macro"),
        })

    # Release this checkpoint before loading the next model generation.
    del (
        model,
        raw_tokens,
        raw_prototypes,
        ensemble_tokens,
        ensemble_text_features,
        ensemble_prototypes,
    )
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# Compare the two text conditions on the same held-out images.
zero_shot_metrics = pl.DataFrame(zero_shot_metric_rows).sort(["model", "method"])
zero_shot_metrics.with_columns(
    pl.col(["accuracy", "macro_f1"]).round(3)
)

In [ ]:
# Build an image browser to inspect model and text-condition disagreements.
zero_shot_details = []
for test_position, image_index in enumerate(test_indices):
    prediction_rows = []
    for method in ZERO_SHOT_METHODS:
        for model_name in MODEL_SPECS:
            probabilities = zero_shot_probabilities[method][model_name]
            top_indices = probabilities[test_position].argsort()[::-1][:5]
            for rank, class_index in enumerate(top_indices, start=1):
                prediction_rows.append([
                    model_name,
                    method,
                    rank,
                    class_names[class_index],
                    f"{probabilities[test_position, class_index]:.3f}",
                ])
    zero_shot_details.append(
        format_details([("Dataset label", labels[image_index])])
        + format_table(
            ["Model", "Text condition", "Rank", "Candidate", "Probability"],
            prediction_rows,
        )
    )

zero_shot_browser = ImageBrowser(
    images=[image_bytes[index] for index in test_indices],
    details=zero_shot_details,
    title="Zero-shot predictions for held-out images",
    random_seed=RANDOM_SEED,
)
zero_shot_browser.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

In [ ]:
# Compare which recorded species are confused under each text condition.
def abbreviate_species_name(name):
    parts = name.split()
    return f"{parts[0][0]}. {' '.join(parts[1:])}" if len(parts) > 1 else name


short_names = [abbreviate_species_name(name) for name in class_names]
test_support = {
    class_name: int(np.sum(y_test == class_name)) for class_name in class_names
}
row_labels = [
    f"{short_name} (n={test_support[class_name]})"
    for class_name, short_name in zip(class_names, short_names)
]


def plot_confusion_matrix(ax, predictions, title):
    macro_f1 = f1_score(y_test, predictions, average="macro")
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        labels=class_names,
        display_labels=short_names,
        normalize="true",
        values_format=".2f",
        xticks_rotation=90,
        colorbar=False,
        ax=ax,
    )
    ax.set_yticklabels(row_labels)
    ax.set_ylabel("Recorded species and test support")
    ax.set_title(f"{title}\nMacro F1 = {macro_f1:.3f}")


fig, axes = plt.subplots(
    len(ZERO_SHOT_METHODS), len(MODEL_SPECS), figsize=(30, 20)
)
for row, method in enumerate(ZERO_SHOT_METHODS):
    for column, model_name in enumerate(MODEL_SPECS):
        plot_confusion_matrix(
            axes[row, column],
            zero_shot_predictions[method][model_name],
            f"{model_name}: {method}",
        )
plt.tight_layout()
plt.show()

# 5/10 Train classifiers on frozen embeddings.

A linear support vector machine, or **linear SVM**, learns separating hyperplanes in an embedding space. It changes the decision boundary, not the encoder.

Every model uses the same training UUIDs and SVM settings. Features are standardized using statistics from the training split because SVM regularization depends on feature scale. The regularization value is fixed before test evaluation rather than selected from the test results.

In [ ]:
# Ask what a linear classifier can extract from each frozen representation.
SVM_C = 0.01  # Fixed before evaluating the test split.
_, svm_training_counts = np.unique(
    labels[train_indices], return_counts=True
)
if len(np.unique(svm_training_counts)) != 1:
    raise ValueError("Expected equal training support for every species")
SVM_SHOTS_PER_SPECIES = int(svm_training_counts[0])


def make_svm():
    # StandardScaler is fit inside the pipeline using training data only.
    return make_pipeline(
        StandardScaler(),
        SVC(kernel="linear", C=SVM_C),
    )


# Fit one classifier per model using the same labeled support images.
svm_predictions = {}
svm_models = {}
comparison_rows = [dict(row) for row in zero_shot_metric_rows]
for model_name, bundle in embedding_bundles.items():
    features = bundle.features
    svm = make_svm()
    svm.fit(features[train_indices], labels[train_indices])
    svm_models[model_name] = svm
    predictions = svm.predict(features[test_indices])
    svm_predictions[model_name] = predictions
    comparison_rows.append({
        "model": model_name,
        "method": "Linear SVM",
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
    })

model_comparison = pl.DataFrame(comparison_rows).sort(["model", "method"])
model_comparison.with_columns(
    pl.col(["accuracy", "macro_f1"]).round(3)
)

In [ ]:
# Place prototype and SVM error patterns on the same class ordering.
comparison_methods = [*ZERO_SHOT_METHODS, "Linear SVM"]
fig, axes = plt.subplots(
    len(comparison_methods), len(MODEL_SPECS), figsize=(30, 29)
)
for row, method in enumerate(comparison_methods):
    for column, model_name in enumerate(MODEL_SPECS):
        predictions = (
            svm_predictions[model_name]
            if method == "Linear SVM"
            else zero_shot_predictions[method][model_name]
        )
        plot_confusion_matrix(
            axes[row, column],
            predictions,
            f"{model_name}: {method}",
        )
plt.tight_layout()
plt.show()

## Inspect errors before collecting more data

Repeated errors between the same species identify distinctions worth investigating. High-margin errors deserve particular attention because the classifier strongly prefers the wrong candidate: they may indicate missing variation, unsuitable training examples, metadata problems, or a genuine visual limitation.

Each gallery image is from the held-out test set. **Dataset label** is the species recorded in its metadata; **SVM prediction** is the classifier output. Select the prediction to open labeled examples of that species in the right panel. Both panels support sequential or random navigation. The decision margin is the difference between the highest and second-highest SVM scores, not a probability. **Pair errors** counts test images with the same directional label-to-prediction error.

In [ ]:
# Rank BioCLIP 2.5 SVM errors for targeted image inspection.
ERROR_MODEL_NAME = "BioCLIP 2.5"
ERROR_GALLERY_SIZE = 12
error_predictions = svm_predictions[ERROR_MODEL_NAME]
decision_scores = svm_models[ERROR_MODEL_NAME].decision_function(
    embedding_bundles[ERROR_MODEL_NAME].features[test_indices]
)
ordered_scores = np.sort(decision_scores, axis=1)
decision_margins = ordered_scores[:, -1] - ordered_scores[:, -2]
error_positions = np.flatnonzero(error_predictions != y_test)

# Summarize recurring recorded-to-predicted species pairs.
error_details = pl.DataFrame(
    {
        "test_position": error_positions,
        "recorded_species": y_test[error_positions],
        "predicted_species": error_predictions[error_positions],
        "decision_margin": decision_margins[error_positions],
    },
    schema={
        "test_position": pl.Int64,
        "recorded_species": pl.String,
        "predicted_species": pl.String,
        "decision_margin": pl.Float64,
    },
)
error_pair_summary = (
    error_details.group_by(["recorded_species", "predicted_species"])
    .agg(
        pl.len().alias("errors"),
        pl.col("decision_margin").mean().alias("mean_margin"),
        pl.col("decision_margin").max().alias("maximum_margin"),
    )
    .sort(["errors", "mean_margin"], descending=True)
)
display(error_pair_summary.with_columns(
    pl.col(["mean_margin", "maximum_margin"]).round(3)
))

# Prioritize recurrent and high-margin mistakes in the gallery.
gallery_errors = (
    error_details.join(
        error_pair_summary.select(
            "recorded_species", "predicted_species", "errors"
        ),
        on=["recorded_species", "predicted_species"],
    )
    .sort(["errors", "decision_margin"], descending=True)
    .head(ERROR_GALLERY_SIZE)
)

if gallery_errors.is_empty():
    print(f"The {ERROR_MODEL_NAME} SVM made no test-set errors.")
else:
    error_rows = gallery_errors.to_dicts()
    error_viewer = ClassificationErrorViewer(
        error_images=[
            image_bytes[test_indices[row["test_position"]]]
            for row in error_rows
        ],
        error_details=[
            format_details([
                ("Dataset label", row["recorded_species"]),
                ("Decision margin", f"{row['decision_margin']:.2f}"),
                ("Errors for this label pair", row["errors"]),
            ])
            for row in error_rows
        ],
        predicted_labels=[
            row["predicted_species"] for row in error_rows
        ],
        reference_images=image_bytes,
        reference_labels=labels,
        title=(
            f"{ERROR_MODEL_NAME} {SVM_SHOTS_PER_SPECIES}-shot SVM errors"
        ),
        random_seed=RANDOM_SEED,
    )
    error_viewer.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

## Vary the number of labeled examples

The **shot count** is the number of labeled training images supplied per species. Results can depend on which images are labeled, so the experiment repeats each shot count 20 times.

Within a repeat, larger training sets retain the examples used at smaller sizes. All models receive the same support UUIDs, and the 100-image test set stays fixed. Curves report mean accuracy with one standard deviation across support-set draws.

In [ ]:
# Ask how classification changes as more labeled images become available.
shot_counts = [1, 2, 5, 10, 20, 30, 40]
repeat_seeds = range(20)

few_shot_rows = evaluate_few_shot_bundles(
    embedding_bundles,
    labels,
    train_indices,
    test_indices,
    class_names,
    shot_counts,
    repeat_seeds,
    make_svm,
    {
        "accuracy": accuracy_score,
        "macro_f1": lambda truth, prediction: f1_score(
            truth, prediction, average="macro"
        ),
    },
)
few_shot_results = pl.DataFrame(few_shot_rows)
# Summarize the repeated support draws at each shot count.
few_shot_summary = (
    few_shot_results.group_by(["model", "shots_per_species"])
    .agg(
        pl.col("accuracy").mean().alias("mean_accuracy"),
        pl.col("accuracy").std().alias("std_accuracy"),
        pl.col("macro_f1").mean().alias("mean_macro_f1"),
    )
    .sort(["model", "shots_per_species"])
)
with pl.Config(tbl_rows=-1):
    display(few_shot_summary.with_columns(
        pl.col([
            "mean_accuracy", "std_accuracy", "mean_macro_f1"
        ]).round(3)
    ))

In [ ]:
# Compare the few-shot curves with each model's prototype baseline.
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.get_cmap("tab10").colors
prototype_accuracy_by_model = {
    row["model"]: row["accuracy"]
    for row in zero_shot_metric_rows
    if row["method"] == PROTOTYPE_METHOD
}

for color, model_name in zip(colors, MODEL_SPECS):
    summary = few_shot_summary.filter(pl.col("model") == model_name)
    x = summary["shots_per_species"].to_numpy()
    mean = summary["mean_accuracy"].to_numpy()
    std = summary["std_accuracy"].to_numpy()
    ax.errorbar(
        x, mean, yerr=std, marker="o", capsize=3,
        color=color, label=f"{model_name} SVM",
    )
    ax.axhline(
        prototype_accuracy_by_model[model_name],
        color=color, linestyle="--", alpha=0.7,
        label=f"{model_name} zero-shot prototype",
    )

ax.set(
    xlabel="Labeled training images per species",
    ylabel="Held-out accuracy",
    title=f"{CHALLENGE_GROUP} label-efficiency comparison",
    xticks=shot_counts,
    ylim=(0, 1),
)
ax.grid(alpha=0.25)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

# 6/10 Adapt the image encoder.

## Align the final visual block with the task

The frozen-embedding results changed the classifier while leaving the image encoder untouched. We now make one constrained change to BioCLIP 2.5: train its final visual transformer block, final normalization, and image projection. The text encoder and earlier visual blocks remain frozen.

The targets are the fixed training-template text prototypes already computed from the original BioCLIP 2.5 text encoder. Cross-entropy encourages each training image embedding to align with its recorded species prototype. No new classification head is added, so the result remains an image embedding that can be compared with text prototypes or supplied to an SVM.

The pretrained CLIP logit scale can make this small supervised loss nearly saturated. Adaptation therefore uses a temperature of 0.10, equivalent to multiplying cosine similarities by 10. This keeps close alternatives involved in the gradient; it does not change the cosine-argmax prediction rule used for evaluation.

The existing 40-image training split is divided into 30 optimization images and 10 validation images per species. The original 10-image test split remains untouched. Validation loss controls early stopping, while validation macro-F1 remains the primary checkpoint criterion.

This is a small, in-distribution adaptation exercise. It can demonstrate how a representation responds to targeted supervision, but it cannot establish performance on new field data.

In [ ]:
# Adapt the final BioCLIP 2.5 visual block toward fixed text prototypes.
from fine_tuning_helpers import (
    configure_last_visual_block,
    load_adaptation_checkpoint,
    load_trainable_state_dict,
    parameter_summary,
    save_adaptation_checkpoint,
    train_last_visual_block,
)
from quantization_helpers import encode_image_collection

# Set the optimization, validation, and checkpoint controls.
ADAPTATION_MODEL_NAME = "BioCLIP 2.5"
ADAPTATION_VALIDATION_PER_SPECIES = 10
ADAPTATION_BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2
ADAPTATION_LEARNING_RATE = 1e-5
ADAPTATION_WEIGHT_DECAY = 0.05
ADAPTATION_PROTOTYPE_TEMPERATURE = 0.10
ADAPTATION_MAXIMUM_EPOCHS = 30
ADAPTATION_PATIENCE = 6
RECOMPUTE_ADAPTATION = False
ADAPTATION_CACHE_DIR = Path("fine_tuning_cache")
ADAPTATION_CACHE_DIR.mkdir(exist_ok=True)

# Divide the original training partition into optimization and validation.
adaptation_train_indices, adaptation_validation_indices = train_test_split(
    train_indices,
    test_size=(
        ADAPTATION_VALIDATION_PER_SPECIES * len(class_names)
    ),
    random_state=RANDOM_SEED + 1,
    stratify=labels[train_indices],
)
adaptation_support = pl.DataFrame({
    "partition": ["optimization", "validation", "test"],
    "images": [
        len(adaptation_train_indices),
        len(adaptation_validation_indices),
        len(test_indices),
    ],
    "images_per_species": [
        len(adaptation_train_indices) // len(class_names),
        len(adaptation_validation_indices) // len(class_names),
        len(test_indices) // len(class_names),
    ],
})
display(adaptation_support)

# Load BioCLIP 2.5 and expose only its final visual block for training.
adaptation_spec = MODEL_SPECS[ADAPTATION_MODEL_NAME]
adaptation_snapshot = snapshot_download(
    repo_id=adaptation_spec["repo_id"],
    revision=adaptation_spec["revision"],
    allow_patterns=[
        "open_clip_config.json",
        adaptation_spec["weight_file"],
    ],
)
adaptation_model, adaptation_training_transform, adaptation_evaluation_transform = (
    open_clip.create_model_and_transforms(
        f"local-dir:{adaptation_snapshot}",
        device=DEVICE,
        precision="fp32",
    )
)
adaptation_model.eval()
adaptation_trainable_names = configure_last_visual_block(adaptation_model)
adaptation_parameter_summary = parameter_summary(adaptation_model)

# Record every choice that determines checkpoint compatibility.
label_to_target = {
    class_name: index for index, class_name in enumerate(class_names)
}
adaptation_targets = np.asarray([
    label_to_target[label] for label in labels
])
adaptation_manifest = {
    "implementation_version": 2,
    "model_repo_id": adaptation_spec["repo_id"],
    "model_revision": adaptation_spec["revision"],
    "method": "final visual block aligned to fixed text prototypes",
    "training_ids_sha256": ids_digest(uuids[adaptation_train_indices]),
    "validation_ids_sha256": ids_digest(
        uuids[adaptation_validation_indices]
    ),
    "prototype_method": PROTOTYPE_METHOD,
    "batch_size": ADAPTATION_BATCH_SIZE,
    "learning_rate": ADAPTATION_LEARNING_RATE,
    "weight_decay": ADAPTATION_WEIGHT_DECAY,
    "prototype_temperature": ADAPTATION_PROTOTYPE_TEMPERATURE,
    "maximum_epochs": ADAPTATION_MAXIMUM_EPOCHS,
    "patience": ADAPTATION_PATIENCE,
    "random_seed": RANDOM_SEED,
}
ADAPTATION_CACHE_TAG = ids_digest([str(adaptation_manifest)])[:12]
adaptation_cache_path = ADAPTATION_CACHE_DIR / (
    f"bioclip2p5_last_visual_block_"
    f"{adaptation_spec['revision'][:8]}_"
    f"{ADAPTATION_CACHE_TAG}.pt"
)

# Reuse a matching checkpoint or optimize a new one against validation loss.
adaptation_checkpoint = None
if not RECOMPUTE_ADAPTATION:
    adaptation_checkpoint = load_adaptation_checkpoint(
        adaptation_cache_path,
        expected_manifest=adaptation_manifest,
    )

if adaptation_checkpoint is None:
    adaptation_checkpoint = train_last_visual_block(
        adaptation_model,
        training_images=[
            image_bytes[index] for index in adaptation_train_indices
        ],
        training_targets=adaptation_targets[adaptation_train_indices],
        validation_images=[
            image_bytes[index] for index in adaptation_validation_indices
        ],
        validation_targets=(
            adaptation_targets[adaptation_validation_indices]
        ),
        training_transform=adaptation_training_transform,
        evaluation_transform=adaptation_evaluation_transform,
        prototypes=zero_shot_prototypes[
            PROTOTYPE_METHOD
        ][ADAPTATION_MODEL_NAME],
        device=DEVICE,
        batch_size=ADAPTATION_BATCH_SIZE,
        learning_rate=ADAPTATION_LEARNING_RATE,
        weight_decay=ADAPTATION_WEIGHT_DECAY,
        prototype_temperature=ADAPTATION_PROTOTYPE_TEMPERATURE,
        maximum_epochs=ADAPTATION_MAXIMUM_EPOCHS,
        patience=ADAPTATION_PATIENCE,
        random_seed=RANDOM_SEED,
    )
    save_adaptation_checkpoint(
        adaptation_cache_path,
        manifest=adaptation_manifest,
        **adaptation_checkpoint,
    )
    print(f"Saved adaptation checkpoint to {adaptation_cache_path}")
else:
    load_trainable_state_dict(
        adaptation_model,
        adaptation_checkpoint["state_dict"],
    )
    print(f"Loaded adaptation checkpoint from {adaptation_cache_path}")

# Report which checkpoint and trainable parameter set will be evaluated.
adaptation_model.eval()
print(f"Best epoch: {adaptation_checkpoint['best_epoch']}")
display(pl.DataFrame([adaptation_parameter_summary]).with_columns(
    pl.col("trainable_percent").round(3)
))
print(f"Trainable tensors: {len(adaptation_trainable_names)}")

## Follow validation performance

Macro-F1 changes only when a predicted class changes, so it can remain flat while useful margins are developing. Validation loss provides a smoother signal for deciding whether training should continue. The retained checkpoint has the highest validation macro-F1, with lower validation loss breaking ties. Epoch 0 is the off-the-shelf model. A widening training-validation loss gap would warn that the final block is fitting the optimization images without improving validation behavior.

In [ ]:
# Ask how optimization and validation behavior changed across epochs.
adaptation_history = pl.DataFrame(adaptation_checkpoint["history"])
display(adaptation_history.with_columns(
    pl.col([
        "training_loss",
        "validation_loss",
        "validation_accuracy",
        "validation_macro_f1",
    ]).round(3)
))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(
    adaptation_history["epoch"],
    adaptation_history["training_loss"],
    marker="o",
)
axes[0].set(
    xlabel="Epoch",
    ylabel="Cross-entropy loss",
    title="Optimization images",
)
axes[1].plot(
    adaptation_history["epoch"],
    adaptation_history["validation_loss"],
    marker="o",
)
axes[1].set(
    xlabel="Epoch",
    ylabel="Cross-entropy loss",
    title="Validation images",
)
axes[2].plot(
    adaptation_history["epoch"],
    adaptation_history["validation_macro_f1"],
    marker="o",
)
axes[2].set(
    xlabel="Epoch",
    ylabel="Macro-F1",
    title="Validation images",
    ylim=(0, 1),
)
for ax in axes:
    ax.axvline(
        adaptation_checkpoint["best_epoch"],
        color="black",
        linestyle="--",
        label="Retained checkpoint",
    )
    ax.grid(alpha=0.25)
axes[2].legend()
plt.tight_layout()
plt.show()

## Evaluate the adapted representation

The adapted image embeddings are evaluated in two ways:

- **Fixed prototype:** compare them with the original BioCLIP 2.5 text prototypes. Because image labels were used during adaptation, this is no longer a zero-shot result.
- **Linear SVM:** refit the same SVM on all 40 labeled training images per species, including the validation images used to select the adapted checkpoint.

Both methods use the original test images. The off-the-shelf rows are repeated here for direct comparison.

In [ ]:
# Re-embed every task image with the retained adapted checkpoint.
adapted_features = encode_image_collection(
    adaptation_model,
    adaptation_evaluation_transform,
    image_bytes,
    batch_size=ADAPTATION_BATCH_SIZE,
    device=DEVICE,
    decode_image=decode_image,
)
adapted_manifest = producer_manifest(
    bundle_id=f"{CHALLENGE_GROUP}:{ADAPTATION_MODEL_NAME}:adapted-fp32",
    model_name=f"{ADAPTATION_MODEL_NAME} adapted",
    repo_id=adaptation_spec["repo_id"],
    revision=adaptation_spec["revision"],
    precision="FP32",
    evidence_type="task-adapted model inference",
    framework="OpenCLIP",
    framework_version=open_clip.__version__,
    preprocessing="OpenCLIP evaluation transform from pinned model config",
    quantization=None,
    dataset={"repo_id": REPO_ID, "subset": CHALLENGE_GROUP},
)
adapted_manifest["adaptation"] = adaptation_manifest
adapted_embedding_bundle = EmbeddingBundle.create(
    uuids,
    adapted_features,
    adapted_manifest,
)

# Repeat fixed-prototype and linear-SVM evaluation on the adapted features.
adapted_similarities = (
    adapted_embedding_bundle.features[test_indices]
    @ zero_shot_prototypes[
        PROTOTYPE_METHOD
    ][ADAPTATION_MODEL_NAME].T
)
adapted_prototype_predictions = np.asarray(class_names)[
    adapted_similarities.argmax(axis=1)
]
adapted_svm = make_svm()
adapted_svm.fit(
    adapted_embedding_bundle.features[train_indices],
    labels[train_indices],
)
adapted_svm_predictions = adapted_svm.predict(
    adapted_embedding_bundle.features[test_indices]
)

# Compare off-the-shelf and adapted representations on the fixed test set.
adaptation_metric_rows = [
    {
        "representation": "Off-the-shelf FP32",
        "method": "Training-template prototype",
        "accuracy": accuracy_score(
            y_test,
            zero_shot_predictions[
                PROTOTYPE_METHOD
            ][ADAPTATION_MODEL_NAME],
        ),
        "macro_f1": f1_score(
            y_test,
            zero_shot_predictions[
                PROTOTYPE_METHOD
            ][ADAPTATION_MODEL_NAME],
            average="macro",
        ),
    },
    {
        "representation": "Off-the-shelf FP32",
        "method": f"{SVM_SHOTS_PER_SPECIES}-shot linear SVM",
        "accuracy": accuracy_score(
            y_test,
            svm_predictions[ADAPTATION_MODEL_NAME],
        ),
        "macro_f1": f1_score(
            y_test,
            svm_predictions[ADAPTATION_MODEL_NAME],
            average="macro",
        ),
    },
    {
        "representation": "Adapted FP32",
        "method": "Fixed training-template prototype",
        "accuracy": accuracy_score(
            y_test,
            adapted_prototype_predictions,
        ),
        "macro_f1": f1_score(
            y_test,
            adapted_prototype_predictions,
            average="macro",
        ),
    },
    {
        "representation": "Adapted FP32",
        "method": f"{SVM_SHOTS_PER_SPECIES}-shot linear SVM",
        "accuracy": accuracy_score(y_test, adapted_svm_predictions),
        "macro_f1": f1_score(
            y_test,
            adapted_svm_predictions,
            average="macro",
        ),
    },
]
adaptation_metrics = pl.DataFrame(adaptation_metric_rows)
display(adaptation_metrics.with_columns(
    pl.col(["accuracy", "macro_f1"]).round(3)
))

In [ ]:
# Inspect which species distinctions changed under adaptation.
adaptation_predictions = [
    (
        zero_shot_predictions[
            PROTOTYPE_METHOD
        ][ADAPTATION_MODEL_NAME],
        "Off-the-shelf: training-template prototype",
    ),
    (
        svm_predictions[ADAPTATION_MODEL_NAME],
        f"Off-the-shelf: {SVM_SHOTS_PER_SPECIES}-shot SVM",
    ),
    (
        adapted_prototype_predictions,
        "Adapted: fixed training-template prototype",
    ),
    (
        adapted_svm_predictions,
        f"Adapted: {SVM_SHOTS_PER_SPECIES}-shot SVM",
    ),
]
fig, axes = plt.subplots(2, 2, figsize=(20, 17))
for ax, (predictions, title) in zip(
    axes.flat,
    adaptation_predictions,
):
    plot_confusion_matrix(ax, predictions, title)
plt.tight_layout()
plt.show()

## Inspect the representation before and after adaptation

Each t-SNE panel is fit independently from the same images. Compare local species overlap within each panel, not absolute position or scale between panels. The task metrics above determine whether a visible change helped classification.

In [ ]:
# Fit separate t-SNE views and link each point to the same source image.
adaptation_tsne_coordinates = {
    "Off-the-shelf BioCLIP 2.5": TSNE(
        metric="euclidean",
        random_state=RANDOM_SEED,
    ).fit_transform(
        embedding_bundles[ADAPTATION_MODEL_NAME].features
    ),
    "Adapted BioCLIP 2.5": TSNE(
        metric="euclidean",
        random_state=RANDOM_SEED,
    ).fit_transform(adapted_embedding_bundle.features),
}
adaptation_embedding_viewer = EmbeddingScatterViewer(
    coordinates_by_panel=adaptation_tsne_coordinates,
    images=image_bytes,
    labels=labels,
    uuids=uuids,
    colors=SPECIES_COLORS,
    title="BioCLIP 2.5 species neighborhoods before and after adaptation",
)
adaptation_embedding_viewer.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

In [ ]:
# Keep the small checkpoint and embedding bundle, then release the
# live ViT-H model before loading models for quantization.
del (
    adaptation_model,
    adaptation_training_transform,
    adaptation_evaluation_transform,
    adapted_features,
)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

# 7/10 Quantize the image encoders.

## Prepare camera-trap images for later analysis

The Lance dataset also contains camera-trap images labeled only as *Peromyscus*. They come from WCS Camera Traps and NACTI, not from the labeled pool used above. MegaDetector v1000 Redwood found an animal in each retained frame at a confidence threshold of 0.30. Their `species` values are null and `seen_in_training` is empty.

We load them now because the quantization pass will create W8A8 embeddings for the same images. They are not used to select classes, fit an SVM, or calculate a score. Detector screening reduces empty frames but does not establish the species or guarantee that every detection is correct.


In [ ]:
# Load the unseen, genus-only camera-trap cohort before quantization.
lila_columns = [
    "uuid",
    "species",
    "seen_in_training",
    "source_dataset",
    "source_id",
    "publisher",
    "source_url",
    "license_name",
    *EMBEDDING_COLUMNS,
    "image",
]
lila_table = lance_ds.scanner(
    filter=LILA_TRIAL_FILTER,
    columns=lila_columns,
).to_table().sort_by([("uuid", "ascending")])

# Preserve alignment among camera metadata, images, and stored embeddings.
lila_metadata = pl.from_arrow(
    lila_table.drop([*EMBEDDING_COLUMNS, "image"])
)
lila_image_bytes = lila_table["image"].combine_chunks().to_pylist()
lila_uuids = np.asarray(lila_metadata["uuid"].to_list(), dtype=np.str_)
lila_raw_embeddings = {
    model_name: np.asarray(
        lila_table[spec["column"]].combine_chunks().values,
        dtype=np.float32,
    ).reshape(-1, spec["dimension"])
    for model_name, spec in MODEL_SPECS.items()
}
del lila_table
gc.collect()

# Enforce the conditions that keep this cohort separate from evaluation labels.
if lila_metadata.is_empty():
    raise ValueError("No screened LILA camera-trap rows were found")
if lila_metadata["species"].null_count() != lila_metadata.height:
    raise ValueError("The camera-trap exercise must not contain species labels")
if any(lila_metadata["seen_in_training"].list.len().fill_null(0)):
    raise ValueError("The camera-trap rows must be marked unseen in training")

# Attach producer metadata to each camera-trap representation.
lila_embedding_bundles = {}
for model_name, values in lila_raw_embeddings.items():
    spec = MODEL_SPECS[model_name]
    lila_embedding_bundles[model_name] = EmbeddingBundle.create(
        lila_uuids,
        values,
        producer_manifest(
            bundle_id=f"LILA-Peromyscus:{model_name}:fp32",
            model_name=model_name,
            repo_id=spec["repo_id"],
            revision=spec["revision"],
            precision="FP32",
            evidence_type="dataset-provided embedding",
            framework="OpenCLIP",
            framework_version=open_clip.__version__,
            preprocessing="OpenCLIP evaluation transform from pinned model config",
            quantization=None,
            dataset={
                "repo_id": REPO_ID,
                "subset": CHALLENGE_GROUP,
                "source_dataset": "lila-bc",
            },
        ),
    )

display(
    lila_metadata.group_by("publisher")
    .len(name="images")
    .sort("publisher")
)
print("Species labels:", lila_metadata["species"].drop_nulls().len())
print("Rows marked as seen in training:", int(
    lila_metadata["seen_in_training"].list.len().fill_null(0).sum()
))


In [ ]:
# Inspect the camera frames with the MegaDetector animal boxes overlaid.
camera_boxes = load_megadetector_boxes(lila_uuids)
camera_overview_browser = ImageBrowser(
    images=lila_image_bytes,
    details=[
        format_details([
            ("Source", row["publisher"]),
            ("Image ID", str(row["uuid"])[:8]),
        ])
        for row in lila_metadata.to_dicts()
    ],
    boxes=camera_boxes,
    title="Genus-only camera-trap images",
    random_seed=RANDOM_SEED,
)
camera_overview_browser.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

## Post-training quantization

**Post-training quantization (PTQ)** changes a trained model's numeric representation without updating its learned weights. No task labels or calibration set are used here.

The selected W8A8 method uses TorchAO's version 2 INT8 tensor path. It stores eligible visual-tower linear weights as per-channel INT8 values. Linear activations are quantized to INT8 dynamically during each call. OpenCLIP also contains raw attention parameters, patch-embedding convolutions, normalization layers, and other tensors that remain FP32. This is therefore a partially quantized image encoder, not an entirely INT8 model. The text tower remains FP32 and its prototypes are reused so the experiment isolates changes to image embeddings.

Each off-the-shelf model follows the same sequence. The adapted BioCLIP 2.5 encoder then follows it separately from its cached fine-tuning checkpoint:

1. Load the FP32 model on the selected device and measure batch-size-one image latency.
2. Quantize eligible linear layers in place and repeat the timing.
3. Encode the labeled images and the genus-only camera-trap images.
4. Pair those image embeddings with the original FP32 text prototypes.
5. Refit each SVM on the new embeddings and repeat evaluation.

Quantized image embeddings and their producer manifests are cached as bundles after a complete run. Set `RECOMPUTE_QUANTIZED_EMBEDDINGS = True` to replace them. The classifier is not cached; it is refit for each bundle and shot count.

The operation counts include matrix multiplication, attention, and convolution. The table reports **nominal GMACs**, or billions of multiply-accumulate operations, once per architecture. One MAC is commonly counted as two FLOPs for floating-point comparisons. Quantization changes numeric type and kernel cost, not the number of MACs. A text-label count represents one fixed-length prompt; a template prototype requires one call per prompt before averaging.


In [ ]:
# Configure one comparable FP32-to-W8A8 experiment for every model.
import os
from pathlib import Path

from quantization_helpers import (
    benchmark_image_encoder,
    encode_image_collection,
    encode_preprocessed_images,
    get_model_size_in_bytes,
    load_or_create_embedding_bundle,
    make_w8a8_config,
    nominal_encoder_gmacs,
    quantize_,
    quantized_weight_parameter_count,
    release_device_memory,
    select_quantization_device,
    w8a8_producer_manifest,
)


QUANT_DEVICE = select_quantization_device(DEVICE)
CPU_THREADS = min(4, os.cpu_count() or 1)
# Batch images through one model. PyTorch uses the selected accelerator without
# duplicating the model in another process.
QUANT_BATCH_SIZE = 8
BENCHMARK_IMAGES = min(4, len(image_bytes))
BENCHMARK_WARMUPS = 1
BENCHMARK_REPEATS = 3
RECOMPUTE_QUANTIZED_EMBEDDINGS = False
QUANT_CACHE_DIR = Path("quantized_embedding_cache")
CAMERA_CACHE_DIR = Path("camera_trap_embedding_cache")
QUANTIZATION_CACHE_TAG = "torchao-int8-v2"
QUANT_CACHE_DIR.mkdir(exist_ok=True)
CAMERA_CACHE_DIR.mkdir(exist_ok=True)

torch.set_num_threads(CPU_THREADS)
if QUANT_DEVICE.type == "cuda":
    print(f"Quantization device: {torch.cuda.get_device_name(QUANT_DEVICE)}")
else:
    print(f"Quantization device: CPU ({torch.get_num_threads()} threads)")
print(
    f"Timing {BENCHMARK_IMAGES} single-image calls after "
    f"{BENCHMARK_WARMUPS} warmup pass(es), repeated {BENCHMARK_REPEATS} times."
)
release_device_memory(QUANT_DEVICE)

In [ ]:
# Quantize each off-the-shelf visual encoder and create matched bundles.
quantized_embedding_bundles = {}
quantized_lila_embedding_bundles = {}
fresh_benchmark_agreement = {}
model_stat_rows = []
timing_rows = []

# Process one checkpoint at a time to bound accelerator and host memory.
for model_name, spec in MODEL_SPECS.items():
    print(f"\nLoading {model_name} on {QUANT_DEVICE.type.upper()}...")
    snapshot = snapshot_download(
        repo_id=spec["repo_id"],
        revision=spec["revision"],
        allow_patterns=["open_clip_config.json", spec["weight_file"]],
    )
    model, _, preprocess = open_clip.create_model_and_transforms(
        f"local-dir:{snapshot}",
        device=QUANT_DEVICE,
        precision="fp32",
    )
    model.eval()

    # Measure the FP32 model before the in-place conversion.
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    image_gmacs, label_gmacs = nominal_encoder_gmacs(model)
    fp32_storage_bytes = get_model_size_in_bytes(model)
    fp32_image_encoder_storage_bytes = get_model_size_in_bytes(model.visual)
    benchmark_tensors = [
        preprocess(decode_image(value)).to(QUANT_DEVICE)
        for value in image_bytes[:BENCHMARK_IMAGES]
    ]
    fp32_seconds_per_image = benchmark_image_encoder(
        model,
        benchmark_tensors,
        device=QUANT_DEVICE,
        warmups=BENCHMARK_WARMUPS,
        repeats=BENCHMARK_REPEATS,
    )
    fp32_benchmark_features = encode_preprocessed_images(
        model, benchmark_tensors
    )
    model_stat_rows.append({
        "model": model_name,
        "precision": "FP32",
        "parameters_millions": total_parameters / 1e6,
        "converted_weight_parameters_millions": 0.0,
        "converted_weight_percent": 0.0,
        "model_storage_mib": fp32_storage_bytes / 1024**2,
        "image_encoder_storage_mib": fp32_image_encoder_storage_bytes / 1024**2,
        "nominal_gmacs_per_image": image_gmacs,
        "nominal_gmacs_per_label": label_gmacs,
    })
    timing_rows.append({
        "model": model_name,
        "precision": "FP32",
        "median_seconds_per_image": fp32_seconds_per_image,
    })

    # Apply the same dynamic W8A8 configuration and repeat the measurements.
    print(f"Quantizing {model_name}...")
    # The conversion is in place, so benchmark FP32 before this line.
    quantize_(model.visual, make_w8a8_config())
    model.eval()
    converted_parameters = quantized_weight_parameter_count(model)
    quantized_storage_bytes = get_model_size_in_bytes(model)
    quantized_image_encoder_storage_bytes = get_model_size_in_bytes(model.visual)
    quantized_seconds_per_image = benchmark_image_encoder(
        model,
        benchmark_tensors,
        device=QUANT_DEVICE,
        warmups=BENCHMARK_WARMUPS,
        repeats=BENCHMARK_REPEATS,
    )
    quantized_benchmark_features = encode_preprocessed_images(
        model, benchmark_tensors
    )
    fresh_benchmark_agreement[model_name] = np.sum(
        fp32_benchmark_features * quantized_benchmark_features, axis=1
    )
    model_stat_rows.append({
        "model": model_name,
        "precision": "W8A8 dynamic PTQ",
        "parameters_millions": total_parameters / 1e6,
        "converted_weight_parameters_millions": converted_parameters / 1e6,
        "converted_weight_percent": 100 * converted_parameters / total_parameters,
        "model_storage_mib": quantized_storage_bytes / 1024**2,
        "image_encoder_storage_mib": quantized_image_encoder_storage_bytes / 1024**2,
        "nominal_gmacs_per_image": image_gmacs,
        "nominal_gmacs_per_label": label_gmacs,
    })
    timing_rows.append({
        "model": model_name,
        "precision": "W8A8 dynamic PTQ",
        "median_seconds_per_image": quantized_seconds_per_image,
    })

    # Load or create W8A8 embeddings for the labeled task.
    cache_name = (
        f"{CHALLENGE_GROUP.lower()}_"
        f"{spec['column']}_{spec['revision'][:8]}_"
        f"{QUANTIZATION_CACHE_TAG}_{QUANT_DEVICE.type}.npz"
    )
    cache_path = QUANT_CACHE_DIR / cache_name
    # UUID validation prevents reuse after the task subset changes. Older
    # feature-only caches are ignored because they lack producer metadata.
    quantized_bundle = load_or_create_embedding_bundle(
        cache_path,
        uuids,
        manifest=w8a8_producer_manifest(
            bundle_id=f"{CHALLENGE_GROUP}:{model_name}:w8a8-dynamic",
            model_name=model_name,
            repo_id=spec["repo_id"],
            revision=spec["revision"],
            dataset={"repo_id": REPO_ID, "subset": CHALLENGE_GROUP},
            device_type=QUANT_DEVICE.type,
        ),
        create_features=lambda: encode_image_collection(
            model,
            preprocess,
            image_bytes,
            batch_size=QUANT_BATCH_SIZE,
            device=QUANT_DEVICE,
            decode_image=decode_image,
        ),
        recompute=RECOMPUTE_QUANTIZED_EMBEDDINGS,
    )
    quantized_embedding_bundles[model_name] = quantized_bundle

    # Apply the same W8A8 producer to the camera-trap cohort.
    lila_cache_name = (
        f"lila_peromyscus_{spec['column']}_{spec['revision'][:8]}_"
        f"{QUANTIZATION_CACHE_TAG}_{QUANT_DEVICE.type}.npz"
    )
    lila_cache_path = CAMERA_CACHE_DIR / lila_cache_name
    quantized_lila_bundle = load_or_create_embedding_bundle(
        lila_cache_path,
        lila_uuids,
        manifest=w8a8_producer_manifest(
            bundle_id=f"LILA-Peromyscus:{model_name}:w8a8-dynamic",
            model_name=model_name,
            repo_id=spec["repo_id"],
            revision=spec["revision"],
            dataset={
                "repo_id": REPO_ID,
                "subset": CHALLENGE_GROUP,
                "source_dataset": "lila-bc",
            },
            device_type=QUANT_DEVICE.type,
        ),
        create_features=lambda: encode_image_collection(
            model,
            preprocess,
            lila_image_bytes,
            batch_size=QUANT_BATCH_SIZE,
            device=QUANT_DEVICE,
            decode_image=decode_image,
        ),
        recompute=RECOMPUTE_QUANTIZED_EMBEDDINGS,
    )
    quantized_lila_embedding_bundles[model_name] = quantized_lila_bundle
    # Only one large model remains resident at a time.
    del (
        model,
        preprocess,
        benchmark_tensors,
        fp32_benchmark_features,
        quantized_benchmark_features,
        quantized_bundle,
        quantized_lila_bundle,
    )
    gc.collect()
    release_device_memory(QUANT_DEVICE)

# Collect the measurements for cross-model comparison.
model_stats = pl.DataFrame(model_stat_rows).sort(["model", "precision"])
timing_results = pl.DataFrame(timing_rows).sort(["model", "precision"])
print("Quantized embedding creation complete.")


## Quantize the adapted BioCLIP 2.5 encoder

The adapted checkpoint contains only the trainable visual tensors. We load the pinned off-the-shelf model, restore those tensors, and apply the same W8A8 method used above. This ordering matters: PTQ describes the adapted weights rather than replacing them with the original checkpoint.

The original FP32 text prototypes remain fixed. Labeled-task and camera-trap embeddings are saved with manifests that record both the adaptation and quantization steps.

In [ ]:
# Restore the adapted checkpoint and subject it to the same W8A8 test.
ADAPTED_MODEL_NAME = f"{ADAPTATION_MODEL_NAME} adapted"
adaptation_snapshot = snapshot_download(
    repo_id=adaptation_spec["repo_id"],
    revision=adaptation_spec["revision"],
    allow_patterns=[
        "open_clip_config.json",
        adaptation_spec["weight_file"],
    ],
)
model, _, preprocess = open_clip.create_model_and_transforms(
    f"local-dir:{adaptation_snapshot}",
    device=QUANT_DEVICE,
    precision="fp32",
)
model.eval()
configure_last_visual_block(model)
restored_adaptation = load_adaptation_checkpoint(
    adaptation_cache_path,
    expected_manifest=adaptation_manifest,
)
if restored_adaptation is None:
    raise ValueError(
        "The matching adaptation checkpoint is missing. "
        "Run section 6 before quantization."
    )
load_trainable_state_dict(model, restored_adaptation["state_dict"])
model.eval()

# Measure and embed with the adapted FP32 model before conversion.
total_parameters = sum(
    parameter.numel() for parameter in model.parameters()
)
image_gmacs, label_gmacs = nominal_encoder_gmacs(model)
fp32_storage_bytes = get_model_size_in_bytes(model)
fp32_image_encoder_storage_bytes = get_model_size_in_bytes(model.visual)
benchmark_tensors = [
    preprocess(decode_image(value)).to(QUANT_DEVICE)
    for value in image_bytes[:BENCHMARK_IMAGES]
]
fp32_seconds_per_image = benchmark_image_encoder(
    model,
    benchmark_tensors,
    device=QUANT_DEVICE,
    warmups=BENCHMARK_WARMUPS,
    repeats=BENCHMARK_REPEATS,
)
fp32_benchmark_features = encode_preprocessed_images(
    model,
    benchmark_tensors,
)
model_stat_rows.append({
    "model": ADAPTED_MODEL_NAME,
    "precision": "FP32",
    "parameters_millions": total_parameters / 1e6,
    "converted_weight_parameters_millions": 0.0,
    "converted_weight_percent": 0.0,
    "model_storage_mib": fp32_storage_bytes / 1024**2,
    "image_encoder_storage_mib": (
        fp32_image_encoder_storage_bytes / 1024**2
    ),
    "nominal_gmacs_per_image": image_gmacs,
    "nominal_gmacs_per_label": label_gmacs,
})
timing_rows.append({
    "model": ADAPTED_MODEL_NAME,
    "precision": "FP32",
    "median_seconds_per_image": fp32_seconds_per_image,
})

# Load or create FP32 camera-trap embeddings from the adapted model.
adapted_lila_fp32_manifest = producer_manifest(
    bundle_id="LILA-Peromyscus:BioCLIP 2.5 adapted:fp32",
    model_name=ADAPTED_MODEL_NAME,
    repo_id=adaptation_spec["repo_id"],
    revision=adaptation_spec["revision"],
    precision="FP32",
    evidence_type="task-adapted model inference",
    framework="OpenCLIP",
    framework_version=open_clip.__version__,
    preprocessing="OpenCLIP evaluation transform from pinned model config",
    quantization=None,
    dataset={
        "repo_id": REPO_ID,
        "subset": CHALLENGE_GROUP,
        "source_dataset": "lila-bc",
    },
)
adapted_lila_fp32_manifest["adaptation"] = adaptation_manifest
adapted_lila_cache_path = CAMERA_CACHE_DIR / (
    "lila_peromyscus_bioclip2p5_adapted_"
    f"{ADAPTATION_CACHE_TAG}_fp32.npz"
)
adapted_lila_embedding_bundle = load_or_create_embedding_bundle(
    adapted_lila_cache_path,
    lila_uuids,
    manifest=adapted_lila_fp32_manifest,
    create_features=lambda: encode_image_collection(
        model,
        preprocess,
        lila_image_bytes,
        batch_size=QUANT_BATCH_SIZE,
        device=QUANT_DEVICE,
        decode_image=decode_image,
    ),
    recompute=RECOMPUTE_QUANTIZED_EMBEDDINGS,
)

# Convert the adapted visual encoder in place and repeat the measurements.
print(f"Quantizing {ADAPTED_MODEL_NAME}...")
for parameter in model.parameters():
    parameter.requires_grad_(False)
quantize_(model.visual, make_w8a8_config())
model.eval()
converted_parameters = quantized_weight_parameter_count(model)
quantized_storage_bytes = get_model_size_in_bytes(model)
quantized_image_encoder_storage_bytes = get_model_size_in_bytes(model.visual)
quantized_seconds_per_image = benchmark_image_encoder(
    model,
    benchmark_tensors,
    device=QUANT_DEVICE,
    warmups=BENCHMARK_WARMUPS,
    repeats=BENCHMARK_REPEATS,
)
quantized_benchmark_features = encode_preprocessed_images(
    model,
    benchmark_tensors,
)
fresh_benchmark_agreement[ADAPTED_MODEL_NAME] = np.sum(
    fp32_benchmark_features * quantized_benchmark_features,
    axis=1,
)
model_stat_rows.append({
    "model": ADAPTED_MODEL_NAME,
    "precision": "W8A8 dynamic PTQ",
    "parameters_millions": total_parameters / 1e6,
    "converted_weight_parameters_millions": (
        converted_parameters / 1e6
    ),
    "converted_weight_percent": (
        100 * converted_parameters / total_parameters
    ),
    "model_storage_mib": quantized_storage_bytes / 1024**2,
    "image_encoder_storage_mib": (
        quantized_image_encoder_storage_bytes / 1024**2
    ),
    "nominal_gmacs_per_image": image_gmacs,
    "nominal_gmacs_per_label": label_gmacs,
})
timing_rows.append({
    "model": ADAPTED_MODEL_NAME,
    "precision": "W8A8 dynamic PTQ",
    "median_seconds_per_image": quantized_seconds_per_image,
})

# Load or create the adapted W8A8 bundle for the labeled task.
adapted_w8a8_manifest = w8a8_producer_manifest(
    bundle_id=f"{CHALLENGE_GROUP}:{ADAPTED_MODEL_NAME}:w8a8-dynamic",
    model_name=ADAPTED_MODEL_NAME,
    repo_id=adaptation_spec["repo_id"],
    revision=adaptation_spec["revision"],
    dataset={"repo_id": REPO_ID, "subset": CHALLENGE_GROUP},
    device_type=QUANT_DEVICE.type,
)
adapted_w8a8_manifest["adaptation"] = adaptation_manifest
adapted_quantized_cache_path = QUANT_CACHE_DIR / (
    "peromyscus_bioclip2p5_adapted_"
    f"{ADAPTATION_CACHE_TAG}_"
    f"{QUANTIZATION_CACHE_TAG}_{QUANT_DEVICE.type}.npz"
)
adapted_quantized_embedding_bundle = load_or_create_embedding_bundle(
    adapted_quantized_cache_path,
    uuids,
    manifest=adapted_w8a8_manifest,
    create_features=lambda: encode_image_collection(
        model,
        preprocess,
        image_bytes,
        batch_size=QUANT_BATCH_SIZE,
        device=QUANT_DEVICE,
        decode_image=decode_image,
    ),
    recompute=RECOMPUTE_QUANTIZED_EMBEDDINGS,
)

# Apply the adapted W8A8 producer to the camera-trap cohort.
adapted_lila_w8a8_manifest = w8a8_producer_manifest(
    bundle_id=f"LILA-Peromyscus:{ADAPTED_MODEL_NAME}:w8a8-dynamic",
    model_name=ADAPTED_MODEL_NAME,
    repo_id=adaptation_spec["repo_id"],
    revision=adaptation_spec["revision"],
    dataset={
        "repo_id": REPO_ID,
        "subset": CHALLENGE_GROUP,
        "source_dataset": "lila-bc",
    },
    device_type=QUANT_DEVICE.type,
)
adapted_lila_w8a8_manifest["adaptation"] = adaptation_manifest
adapted_lila_w8a8_cache_path = CAMERA_CACHE_DIR / (
    "lila_peromyscus_bioclip2p5_adapted_"
    f"{ADAPTATION_CACHE_TAG}_"
    f"{QUANTIZATION_CACHE_TAG}_{QUANT_DEVICE.type}.npz"
)
adapted_quantized_lila_embedding_bundle = (
    load_or_create_embedding_bundle(
        adapted_lila_w8a8_cache_path,
        lila_uuids,
        manifest=adapted_lila_w8a8_manifest,
        create_features=lambda: encode_image_collection(
            model,
            preprocess,
            lila_image_bytes,
            batch_size=QUANT_BATCH_SIZE,
            device=QUANT_DEVICE,
            decode_image=decode_image,
        ),
        recompute=RECOMPUTE_QUANTIZED_EMBEDDINGS,
    )
)

# Release the adapted checkpoint after both cohorts are embedded.
del (
    model,
    preprocess,
    benchmark_tensors,
    fp32_benchmark_features,
    quantized_benchmark_features,
)
gc.collect()
release_device_memory(QUANT_DEVICE)

model_stats = pl.DataFrame(model_stat_rows).sort(["model", "precision"])
timing_results = pl.DataFrame(timing_rows).sort(["model", "precision"])
print("Adapted quantized embedding creation complete.")

## Model properties and measured inference latency

The following displays separate architecture, storage, and timing concerns:

- **Parameters** count learned values. PTQ does not remove parameters.
- **Converted weights** are the parameters represented as INT8 by this method.
- **Full model storage** includes the image and text encoders. **Image encoder storage** is the relevant lower bound when text prototypes are computed before deployment. Neither includes Python, temporary activations, or total process memory.
- **Nominal GMACs** describe architecture-level work. They remain constant after quantization.
- **Measured speedup** is FP32 time divided by W8A8 time. Values above 1 mean W8A8 was faster.

Timing excludes image decoding and preprocessing. It uses repeated batch-size-one calls on the selected device, so it supports comparisons within this kernel rather than predictions for another device. The subsequent full embedding pass uses batches of eight for better throughput.

In [ ]:
# Ask what changed in provenance, storage, and measured inference time.
bundle_inventory = pl.DataFrame([
    {
        "bundle_id": bundle.manifest["bundle_id"],
        "model": bundle.manifest["model"]["name"],
        "precision": bundle.manifest["precision"],
        "evidence_type": bundle.manifest["evidence_type"],
        "backend": bundle.manifest["backend"],
        "delegation": bundle.manifest["delegation"],
        "target_latency_comparable": (
            bundle.manifest.get("runtime") or {}
        ).get("target_runtime_latency_comparable", False),
    }
    for bundle in [
        *embedding_bundles.values(),
        *quantized_embedding_bundles.values(),
        adapted_embedding_bundle,
        adapted_quantized_embedding_bundle,
    ]
])
print("Embedding producers")
display(bundle_inventory)

# Separate architecture cost from the storage changed by quantization.
architecture_stats = (
    model_stats.filter(pl.col("precision") == "FP32")
    .select(
        "model",
        pl.col("parameters_millions").alias("params_m"),
        pl.col("nominal_gmacs_per_image").alias("gmacs_per_image"),
        pl.col("nominal_gmacs_per_label").alias("gmacs_per_label"),
    )
    .sort("model")
)

fp32_storage = (
    model_stats.filter(pl.col("precision") == "FP32")
    .select(
        "model",
        pl.col("model_storage_mib").alias("fp32_storage_mib"),
        pl.col("image_encoder_storage_mib").alias("fp32_image_encoder_mib"),
    )
)
quantized_storage = (
    model_stats.filter(pl.col("precision") == "W8A8 dynamic PTQ")
    .select(
        "model",
        pl.col("converted_weight_parameters_millions").alias("converted_weight_params_m"),
        pl.col("converted_weight_percent").alias("converted_weight_pct"),
        pl.col("model_storage_mib").alias("w8a8_storage_mib"),
        pl.col("image_encoder_storage_mib").alias("w8a8_image_encoder_mib"),
    )
)
quantization_stats = (
    fp32_storage.join(quantized_storage, on="model")
    .with_columns(
        (
            100
            * (1 - pl.col("w8a8_storage_mib") / pl.col("fp32_storage_mib"))
        ).alias("full_storage_reduction_pct"),
        (
            100
            * (
                1
                - pl.col("w8a8_image_encoder_mib")
                / pl.col("fp32_image_encoder_mib")
            )
        ).alias("image_encoder_reduction_pct"),
    )
    .sort("model")
)

# Compare timing only within the processor and runtime used in this session.
timing_comparison = (
    timing_results.pivot(
        on="precision",
        index="model",
        values="median_seconds_per_image",
    )
    .rename({
        "FP32": "fp32_s_per_image",
        "W8A8 dynamic PTQ": "w8a8_s_per_image",
    })
    .with_columns(
        (
            pl.col("fp32_s_per_image")
            / pl.col("w8a8_s_per_image")
        ).alias("speedup_x"),
        (
            100
            * (
                1
                - pl.col("w8a8_s_per_image")
                / pl.col("fp32_s_per_image")
            )
        ).alias("latency_reduction_pct"),
    )
    .sort("model")
)

print("Architecture")
display(architecture_stats.with_columns(pl.exclude("model").round(3)))
print("Quantized storage")
display(quantization_stats.with_columns(pl.exclude("model").round(2)))
print(f"Measured {QUANT_DEVICE.type.upper()} latency")
display(timing_comparison.with_columns(pl.exclude("model").round(3)))

## Back-of-the-napkin deployment check

The model tables describe two separate resource questions:

- **Will it fit?** For a fixed classifier, start with `image_encoder_storage_mib`. Text prototypes or SVM weights are small and can be saved with the application, so the text encoder does not need to run at the edge. Tensor storage is still only a lower bound for RAM use. The operating system, Python, PyTorch, decoded images, temporary activations, and the application also need memory.
- **How fast might it run?** A **MAC** is one multiplication followed by accumulation into a sum. A **GMAC** is one billion MACs. For FP32 comparisons, one MAC is commonly reported as two FLOPs.

`gmacs_per_image` estimates the work required to encode one image. `gmacs_per_label` estimates one text-encoder call. Text prototypes can usually be computed once and stored, so label encoding is not normally paid for every image.

For assumed-throughput arithmetic:

```text
seconds per image ~= GMAC per image / sustained effective GMAC per second
images per second ~= sustained effective GMAC per second / GMAC per image
```

This calculation is not a latency measurement or prediction unless the throughput value was measured with the intended model, device, and runtime. Use measured sustained MAC throughput rather than a processor's advertised peak. Enter separate FP32 and W8A8 rates: quantization does not change the MAC count, but it can change the time required for each MAC. W8A8 models still contain FP32 operations, and an ARM device such as a Raspberry Pi may use different kernels from this server.

For example, dividing 17.5 GMAC by an assumed 10 GMAC/s gives 1.75 seconds of compute. That value becomes informative only when 10 GMAC/s is a relevant measurement for this model and runtime. Image loading, preprocessing, memory traffic, and unsupported operations make end-to-end latency different.

In [ ]:
# Ask whether each image encoder fits a sample memory and compute budget.
# Example target budget. Replace throughput with a measurement from the
# intended model, device, and inference runtime.
TARGET_RAM_GIB = 8.0
SYSTEM_AND_APP_RESERVE_GIB = 2.0
ASSUMED_EFFECTIVE_GMACS_PER_SECOND = {
    "FP32": 10.0,
    "W8A8 dynamic PTQ": 10.0,
}  # Illustrative values, not an RPi benchmark.

usable_model_mib = (
    TARGET_RAM_GIB - SYSTEM_AND_APP_RESERVE_GIB
) * 1024
deployment_estimates = (
    model_stats.select(
        "model",
        "precision",
        pl.col("image_encoder_storage_mib").alias("image_encoder_storage_mib"),
        pl.col("nominal_gmacs_per_image").alias("gmacs_per_image"),
    )
    .with_columns(
        pl.col("precision")
        .replace_strict(ASSUMED_EFFECTIVE_GMACS_PER_SECOND)
        .alias("assumed_gmacs_per_second"),
        (
            pl.col("image_encoder_storage_mib") <= usable_model_mib
        ).alias("image_encoder_fits_budget"),
    )
    .with_columns(
        (
            pl.col("gmacs_per_image")
            / pl.col("assumed_gmacs_per_second")
        ).alias("compute_seconds_at_assumed_throughput"),
        (
            pl.col("assumed_gmacs_per_second")
            / pl.col("gmacs_per_image")
        ).alias("compute_images_per_second_at_assumed_throughput"),
    )
    .sort(["model", "precision"])
)

print(
    f"RAM budget for model tensors: {usable_model_mib / 1024:.1f} GiB "
    f"after a {SYSTEM_AND_APP_RESERVE_GIB:.1f} GiB reserve"
)
deployment_estimates.with_columns(
    pl.col([
        "image_encoder_storage_mib",
        "assumed_gmacs_per_second",
        "gmacs_per_image",
        "compute_seconds_at_assumed_throughput",
        "compute_images_per_second_at_assumed_throughput",
    ]).round(3)
)

## Embedding agreement

Cosine similarity measures direction agreement:

- 1 means the vectors point in the same direction.
- 0 means they are orthogonal.
- -1 means they point in opposite directions.

`fresh_benchmark_mean_cosine` isolates quantization by comparing FP32 and W8A8 outputs from the same freshly loaded model on the timed images. The remaining columns compare all W8A8 embeddings with the stored FP32 vectors for the same model and UUID.

High agreement is useful, but task metrics determine whether the remaining differences affect this species classifier.

In [ ]:
# Measure how closely each W8A8 embedding follows its FP32 counterpart.
embedding_agreement_rows = []
for model_name in MODEL_SPECS:
    cosine_similarity = np.sum(
        embedding_bundles[model_name].features
        * quantized_embedding_bundles[model_name].features,
        axis=1,
    )
    embedding_agreement_rows.append({
        "model": model_name,
        "fresh_benchmark_mean_cosine": float(
            fresh_benchmark_agreement[model_name].mean()
        ),
        "mean_cosine_similarity": float(cosine_similarity.mean()),
        "minimum_cosine_similarity": float(cosine_similarity.min()),
        "p05_cosine_similarity": float(
            np.quantile(cosine_similarity, 0.05)
        ),
        "median_cosine_similarity": float(np.median(cosine_similarity)),
    })

# Apply the same agreement calculation to the adapted model.
adapted_cosine_similarity = np.sum(
    adapted_embedding_bundle.features
    * adapted_quantized_embedding_bundle.features,
    axis=1,
)
embedding_agreement_rows.append({
    "model": ADAPTED_MODEL_NAME,
    "fresh_benchmark_mean_cosine": float(
        fresh_benchmark_agreement[ADAPTED_MODEL_NAME].mean()
    ),
    "mean_cosine_similarity": float(
        adapted_cosine_similarity.mean()
    ),
    "minimum_cosine_similarity": float(
        adapted_cosine_similarity.min()
    ),
    "p05_cosine_similarity": float(
        np.quantile(adapted_cosine_similarity, 0.05)
    ),
    "median_cosine_similarity": float(
        np.median(adapted_cosine_similarity)
    ),
})

embedding_agreement = pl.DataFrame(embedding_agreement_rows).sort("model")
embedding_agreement.with_columns(pl.exclude("model").round(4))

## Species structure after quantization

FP32 and W8A8 are fit separately using the same image IDs, t-SNE settings, and random seed. The FP32 panel reuses the coordinates shown earlier. Compare whether each species remains compact, fragmented, or intermixed within its panel. Coordinates, orientation, scale, and apparent distances are not comparable between panels. Selecting a marker highlights the same image in both panels and displays it below each plot. Classification metrics determine whether any local changes matter for the task.

In [ ]:
# Fit the W8A8 t-SNE independently and link matching UUIDs across panels.
TSNE_QUANT_MODEL = "BioCLIP 2.5"
w8a8_tsne_features = quantized_embedding_bundles[TSNE_QUANT_MODEL].features
precision_coordinates = {
    "FP32": tsne_coordinates[TSNE_QUANT_MODEL],
    "W8A8 dynamic PTQ": TSNE(
        metric="euclidean",
        random_state=RANDOM_SEED,
    ).fit_transform(w8a8_tsne_features),
}

quantized_space_viewer = EmbeddingScatterViewer(
    coordinates_by_panel=precision_coordinates,
    images=image_bytes,
    labels=labels,
    uuids=uuids,
    colors=SPECIES_COLORS,
    title=(
        f"{TSNE_QUANT_MODEL} species neighborhoods: "
        "independent t-SNE fits"
    ),
    link_panels=True,
)
quantized_space_viewer.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

# 8/10 Repeat the scientific task.

## Repeat zero-shot and SVM evaluation

For the off-the-shelf zero-shot classification, W8A8 image embeddings are compared with the original FP32 text prototypes. For the supervised condition, a new SVM is fit on the W8A8 image embeddings using the original UUID split and classifier settings.

Small metric increases are possible because quantization changes embedding geometry and the SVM is retrained in that new space. A higher score on this fixed 100-image test set should be described as an observed result, not evidence that quantization generally improves the model.

In [ ]:
# Repeat prototype and SVM evaluation using W8A8 image embeddings.
quantized_zero_shot_predictions = {
    method: {} for method in ZERO_SHOT_METHODS
}
quantized_svm_predictions = {}
quantized_svm_models = {}
quantized_metric_rows = []

# Keep the text prototypes, labels, support images, and test images fixed.
for model_name in MODEL_SPECS:
    features = quantized_embedding_bundles[model_name].features
    for method in ZERO_SHOT_METHODS:
        prototypes = zero_shot_prototypes[method][model_name]
        similarities = features[test_indices] @ prototypes.T
        predictions = np.asarray(class_names)[similarities.argmax(axis=1)]
        quantized_zero_shot_predictions[method][model_name] = predictions
        quantized_metric_rows.append({
            "model": model_name,
            "precision": "W8A8 dynamic PTQ",
            "method": method,
            "accuracy": accuracy_score(y_test, predictions),
            "macro_f1": f1_score(y_test, predictions, average="macro"),
        })

    svm = make_svm()
    svm.fit(features[train_indices], labels[train_indices])
    quantized_svm_models[model_name] = svm
    predictions = svm.predict(features[test_indices])
    quantized_svm_predictions[model_name] = predictions
    quantized_metric_rows.append({
        "model": model_name,
        "precision": "W8A8 dynamic PTQ",
        "method": "Linear SVM",
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
    })

# Place FP32 and W8A8 scores in one paired table.
full_precision_metric_rows = [
    {**row, "precision": "FP32"} for row in comparison_rows
]
precision_comparison = pl.DataFrame(
    [*full_precision_metric_rows, *quantized_metric_rows]
).sort(["model", "method", "precision"])
display(
    precision_comparison.with_columns(
        pl.col(["accuracy", "macro_f1"]).round(3)
    )
)

# Ask how macro-F1 changed for each model and classification method.
macro_f1_change = (
    precision_comparison.select(
        "model", "method", "precision", "macro_f1"
    )
    .pivot(
        on="precision",
        index=["model", "method"],
        values="macro_f1",
    )
    .rename({
        "FP32": "fp32_macro_f1",
        "W8A8 dynamic PTQ": "w8a8_macro_f1",
    })
    .with_columns(
        (
            pl.col("w8a8_macro_f1") - pl.col("fp32_macro_f1")
        ).alias("w8a8_minus_fp32")
    )
    .sort(["model", "method"])
)
print("Macro F1 change on the fixed test set")
macro_f1_change.with_columns(
    pl.exclude(["model", "method"]).round(3)
)

## Repeat the adapted-model evaluation after quantization

The W8A8 adapted embeddings are compared with the same fixed text prototypes, and a new 40-shot SVM is fit in the quantized embedding space. This isolates the deployment change from the earlier adaptation step.

In [ ]:
# Repeat fixed-prototype and SVM evaluation for adapted W8A8 features.
adapted_quantized_features = (
    adapted_quantized_embedding_bundle.features
)
adapted_quantized_similarities = (
    adapted_quantized_features[test_indices]
    @ zero_shot_prototypes[
        PROTOTYPE_METHOD
    ][ADAPTATION_MODEL_NAME].T
)
adapted_quantized_prototype_predictions = np.asarray(class_names)[
    adapted_quantized_similarities.argmax(axis=1)
]
adapted_quantized_svm = make_svm()
adapted_quantized_svm.fit(
    adapted_quantized_features[train_indices],
    labels[train_indices],
)
adapted_quantized_svm_predictions = adapted_quantized_svm.predict(
    adapted_quantized_features[test_indices]
)

# Compare adapted FP32 and W8A8 on the same test images.
adapted_precision_rows = [
    {
        "precision": "FP32",
        "method": "Fixed training-template prototype",
        "accuracy": accuracy_score(
            y_test,
            adapted_prototype_predictions,
        ),
        "macro_f1": f1_score(
            y_test,
            adapted_prototype_predictions,
            average="macro",
        ),
    },
    {
        "precision": "FP32",
        "method": f"{SVM_SHOTS_PER_SPECIES}-shot linear SVM",
        "accuracy": accuracy_score(y_test, adapted_svm_predictions),
        "macro_f1": f1_score(
            y_test,
            adapted_svm_predictions,
            average="macro",
        ),
    },
    {
        "precision": "W8A8 dynamic PTQ",
        "method": "Fixed training-template prototype",
        "accuracy": accuracy_score(
            y_test,
            adapted_quantized_prototype_predictions,
        ),
        "macro_f1": f1_score(
            y_test,
            adapted_quantized_prototype_predictions,
            average="macro",
        ),
    },
    {
        "precision": "W8A8 dynamic PTQ",
        "method": f"{SVM_SHOTS_PER_SPECIES}-shot linear SVM",
        "accuracy": accuracy_score(
            y_test,
            adapted_quantized_svm_predictions,
        ),
        "macro_f1": f1_score(
            y_test,
            adapted_quantized_svm_predictions,
            average="macro",
        ),
    },
]
adapted_precision_comparison = pl.DataFrame(
    adapted_precision_rows
).sort(["method", "precision"])
display(adapted_precision_comparison.with_columns(
    pl.col(["accuracy", "macro_f1"]).round(3)
))

# Ask whether quantization changed either adapted-model interface.
adapted_macro_f1_change = (
    adapted_precision_comparison.select(
        "method", "precision", "macro_f1"
    )
    .pivot(
        on="precision",
        index="method",
        values="macro_f1",
    )
    .rename({
        "FP32": "fp32_macro_f1",
        "W8A8 dynamic PTQ": "w8a8_macro_f1",
    })
    .with_columns(
        (
            pl.col("w8a8_macro_f1")
            - pl.col("fp32_macro_f1")
        ).alias("w8a8_minus_fp32")
    )
)
print("Adapted-model macro-F1 change")
display(adapted_macro_f1_change.with_columns(
    pl.exclude("method").round(3)
))

In [ ]:
# Inspect adapted-model error patterns before and after W8A8.
adapted_precision_predictions = [
    (
        adapted_prototype_predictions,
        "Adapted FP32: fixed prototype",
    ),
    (
        adapted_quantized_prototype_predictions,
        "Adapted W8A8: fixed prototype",
    ),
    (
        adapted_svm_predictions,
        f"Adapted FP32: {SVM_SHOTS_PER_SPECIES}-shot SVM",
    ),
    (
        adapted_quantized_svm_predictions,
        f"Adapted W8A8: {SVM_SHOTS_PER_SPECIES}-shot SVM",
    ),
]
fig, axes = plt.subplots(2, 2, figsize=(20, 17))
for ax, (predictions, title) in zip(
    axes.flat,
    adapted_precision_predictions,
):
    plot_confusion_matrix(ax, predictions, title)
plt.tight_layout()
plt.show()

In [ ]:
# Inspect off-the-shelf W8A8 error patterns across models and methods.
fig, axes = plt.subplots(
    len(comparison_methods), len(MODEL_SPECS), figsize=(30, 29)
)
for row, method in enumerate(comparison_methods):
    for column, model_name in enumerate(MODEL_SPECS):
        predictions = (
            quantized_svm_predictions[model_name]
            if method == "Linear SVM"
            else quantized_zero_shot_predictions[method][model_name]
        )
        plot_confusion_matrix(
            axes[row, column],
            predictions,
            f"{model_name} W8A8: {method}",
        )
plt.tight_layout()
plt.show()

## Which predictions changed after quantization?

A prediction flip shows where the deployment change reaches the task decision boundary. Regressions, improvements, and changes between two incorrect labels are separated below. Embedding cosine similarity indicates how small the representation change was, while the images show whether the affected cases share visible conditions worth investigating.

In [ ]:
# Find test images whose BioCLIP 2.5 SVM prediction changes after W8A8.
FLIP_MODEL_NAME = "BioCLIP 2.5"
FLIP_GALLERY_SIZE = 12
fp32_predictions = svm_predictions[FLIP_MODEL_NAME]
w8a8_predictions = quantized_svm_predictions[FLIP_MODEL_NAME]
flip_positions = np.flatnonzero(fp32_predictions != w8a8_predictions)
embedding_cosines = np.sum(
    embedding_bundles[FLIP_MODEL_NAME].features[test_indices]
    * quantized_embedding_bundles[FLIP_MODEL_NAME].features[test_indices],
    axis=1,
)

# Label each changed prediction by its effect on the recorded test label.
flip_rows = []
for position in flip_positions:
    fp32_correct = fp32_predictions[position] == y_test[position]
    w8a8_correct = w8a8_predictions[position] == y_test[position]
    if fp32_correct and not w8a8_correct:
        outcome = "regression"
    elif w8a8_correct and not fp32_correct:
        outcome = "improvement"
    else:
        outcome = "changed, both incorrect"
    flip_rows.append({
        "test_position": int(position),
        "recorded_species": y_test[position],
        "fp32_prediction": fp32_predictions[position],
        "w8a8_prediction": w8a8_predictions[position],
        "outcome": outcome,
        "embedding_cosine": float(embedding_cosines[position]),
    })
flip_details = pl.DataFrame(
    flip_rows,
    schema={
        "test_position": pl.Int64,
        "recorded_species": pl.String,
        "fp32_prediction": pl.String,
        "w8a8_prediction": pl.String,
        "outcome": pl.String,
        "embedding_cosine": pl.Float64,
    },
)
# Inspect the changed cases with the lowest embedding agreement first.
if flip_details.is_empty():
    print("No BioCLIP 2.5 SVM predictions changed after quantization.")
else:
    display(
        flip_details.group_by("outcome")
        .len(name="images")
        .sort("outcome")
    )
    outcome_order = {
        "regression": 0,
        "improvement": 1,
        "changed, both incorrect": 2,
    }
    gallery_flips = (
        flip_details.with_columns(
            pl.col("outcome").replace_strict(outcome_order).alias("order")
        )
        .sort(["order", "embedding_cosine"])
        .head(FLIP_GALLERY_SIZE)
    )
    flip_gallery_rows = gallery_flips.to_dicts()
    flip_browser = ImageBrowser(
        images=[
            image_bytes[test_indices[row["test_position"]]]
            for row in flip_gallery_rows
        ],
        details=[
            format_details([
                ("Outcome", row["outcome"]),
                ("Dataset label", row["recorded_species"]),
                ("FP32 prediction", row["fp32_prediction"]),
                ("W8A8 prediction", row["w8a8_prediction"]),
                ("Embedding cosine", f"{row['embedding_cosine']:.4f}"),
            ])
            for row in flip_gallery_rows
        ],
        title=f"{FLIP_MODEL_NAME} prediction changes after W8A8",
        random_seed=RANDOM_SEED,
    )
    flip_browser.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

## Number of labeled examples after quantization

The support-set experiment is repeated with W8A8 embeddings. Each panel compares FP32 and W8A8 for one model using the same nested support examples and fixed test set.

This checks whether quantization changes the amount of labeled data needed for the downstream task, not only the final 40-shot SVM score.

In [ ]:
# Repeat the nested few-shot experiment with the W8A8 bundles.
quantized_few_shot_rows = evaluate_few_shot_bundles(
    quantized_embedding_bundles,
    labels,
    train_indices,
    test_indices,
    class_names,
    shot_counts,
    repeat_seeds,
    make_svm,
    {
        "accuracy": accuracy_score,
        "macro_f1": lambda truth, prediction: f1_score(
            truth, prediction, average="macro"
        ),
    },
)

# Summarize the repeated support draws at each shot count.
quantized_few_shot_results = pl.DataFrame(quantized_few_shot_rows)
quantized_few_shot_summary = (
    quantized_few_shot_results.group_by(
        ["model", "shots_per_species"]
    )
    .agg(
        pl.col("accuracy").mean().alias("mean_accuracy"),
        pl.col("accuracy").std().alias("std_accuracy"),
        pl.col("macro_f1").mean().alias("mean_macro_f1"),
    )
    .sort(["model", "shots_per_species"])
)
with pl.Config(tbl_rows=-1):
    display(quantized_few_shot_summary.with_columns(
        pl.col([
            "mean_accuracy", "std_accuracy", "mean_macro_f1"
        ]).round(3)
    ))

In [ ]:
# Compare label efficiency before and after W8A8 for each model.
fig, axes = plt.subplots(
    1, len(MODEL_SPECS), figsize=(18, 5), sharex=True, sharey=True
)
for ax, model_name in zip(axes, MODEL_SPECS):
    for summary, label, color in [
        (few_shot_summary, "FP32", "tab:blue"),
        (quantized_few_shot_summary, "W8A8 dynamic PTQ", "tab:orange"),
    ]:
        model_summary = summary.filter(pl.col("model") == model_name)
        ax.errorbar(
            model_summary["shots_per_species"].to_numpy(),
            model_summary["mean_accuracy"].to_numpy(),
            yerr=model_summary["std_accuracy"].to_numpy(),
            marker="o",
            capsize=3,
            color=color,
            label=label,
        )
    ax.set(
        title=model_name,
        xlabel="Labeled training images per species",
        xticks=shot_counts,
        ylim=(0, 1),
    )
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Held-out accuracy")
axes[-1].legend()
fig.suptitle(f"{CHALLENGE_GROUP}: FP32 and W8A8 label efficiency")
plt.tight_layout()
plt.show()

# 9/10 Examine genus-only camera-trap images.

## Which species are these camera-trap mice?

The source records identify these images only as *Peromyscus*. We ask the three off-the-shelf model generations and the adapted BioCLIP 2.5 model for two species predictions at FP32 and W8A8:

- The **training-template prototype** uses the complete text ensemble shown earlier. The adapted model uses the original BioCLIP 2.5 text prototypes.
- The **40-shot linear SVM** is refit for each embedding representation.

The candidate set contains the same ten species as the labeled task. A prediction outside that set is impossible, and there is no species ground truth for an accuracy or F1 score. The margins below show separation between the first and second choices, not probabilities.


## Inspect one image at a time

Move through the genus-only camera-trap images, run all sixteen closed-set predictions, and compare the suggested species with labeled reference images. The thin red outline is the stored MegaDetector animal detection used during preprocessing; the notebook does not rerun the detector. Clicking a species button repeatedly replaces the reference image with another example from the labeled Lance subset.


In [ ]:
# Assemble every model, precision, and classifier for interactive comparison.
from interactive_camera_trap import (
    CameraTrapViewer,
    build_prediction_sources,
)

# Pair each camera-trap embedding bundle with its matching fitted classifier.
CAMERA_MODEL_NAMES = [*MODEL_SPECS, ADAPTED_MODEL_NAME]
camera_prototypes = {
    **zero_shot_prototypes[PROTOTYPE_METHOD],
    ADAPTED_MODEL_NAME: zero_shot_prototypes[
        PROTOTYPE_METHOD
    ][ADAPTATION_MODEL_NAME],
}
camera_fp32_bundles = {
    **lila_embedding_bundles,
    ADAPTED_MODEL_NAME: adapted_lila_embedding_bundle,
}
camera_fp32_svms = {
    **svm_models,
    ADAPTED_MODEL_NAME: adapted_svm,
}
camera_w8a8_bundles = {
    **quantized_lila_embedding_bundles,
    ADAPTED_MODEL_NAME: adapted_quantized_lila_embedding_bundle,
}
camera_w8a8_svms = {
    **quantized_svm_models,
    ADAPTED_MODEL_NAME: adapted_quantized_svm,
}
camera_variants = {
    "FP32": (camera_fp32_bundles, camera_fp32_svms),
    "W8A8 dynamic PTQ": (
        camera_w8a8_bundles,
        camera_w8a8_svms,
    ),
}
# Connect all prediction sources to one image-at-a-time viewer.
camera_prediction_sources = build_prediction_sources(
    model_names=CAMERA_MODEL_NAMES,
    class_names=class_names,
    prototype_method=PROTOTYPE_METHOD,
    prototypes=camera_prototypes,
    variants=camera_variants,
)
camera_viewer = CameraTrapViewer(
    camera_images=lila_image_bytes,
    camera_metadata=lila_metadata.select(
        "uuid", "publisher", "source_url"
    ).to_dicts(),
    camera_boxes=camera_boxes,
    prediction_sources=camera_prediction_sources,
    reference_images=image_bytes,
    reference_labels=labels,
    random_seed=RANDOM_SEED,
)
camera_viewer.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.


In [ ]:
# Calculate all camera-trap predictions for aggregate comparisons.
def top_two_margin(scores):
    ordered = np.sort(np.asarray(scores), axis=1)
    return ordered[:, -1] - ordered[:, -2]


# Record prototype and SVM choices plus their first-to-second margins.
camera_prediction_rows = []
prediction_arrays = {}
for precision, (bundles, fitted_svms) in camera_variants.items():
    for model_name in CAMERA_MODEL_NAMES:
        features = bundles[model_name].features

        similarities = (
            features @ camera_prototypes[model_name].T
        )
        zero_shot = np.asarray(class_names)[similarities.argmax(axis=1)]
        prediction_arrays[
            (model_name, precision, PROTOTYPE_METHOD)
        ] = zero_shot

        svm_scores = fitted_svms[model_name].decision_function(features)
        svm_prediction = fitted_svms[model_name].predict(features)
        prediction_arrays[
            (model_name, precision, "40-shot linear SVM")
        ] = svm_prediction

        for method, predictions, margins in [
            (
                PROTOTYPE_METHOD,
                zero_shot,
                top_two_margin(similarities),
            ),
            (
                "40-shot linear SVM",
                svm_prediction,
                top_two_margin(svm_scores),
            ),
        ]:
            for index, predicted_species in enumerate(predictions):
                camera_prediction_rows.append({
                    "uuid": str(lila_uuids[index]),
                    "publisher": lila_metadata["publisher"][index],
                    "source_url": lila_metadata["source_url"][index],
                    "model": model_name,
                    "precision": precision,
                    "method": method,
                    "predicted_species": str(predicted_species),
                    "margin": float(margins[index]),
                })

camera_predictions = pl.DataFrame(camera_prediction_rows)
print(
    f"{camera_predictions.height:,} predictions for "
    f"{len(lila_uuids):,} images"
)


In [ ]:
# Ask where methods and precision variants choose the same species.
agreement_rows = []
for model_name in CAMERA_MODEL_NAMES:
    for precision in camera_variants:
        prototype = prediction_arrays[
            (model_name, precision, PROTOTYPE_METHOD)
        ]
        svm = prediction_arrays[
            (model_name, precision, "40-shot linear SVM")
        ]
        agreement_rows.append({
            "model": model_name,
            "precision": precision,
            "prototype_svm_agreement": float(np.mean(prototype == svm)),
        })
    for method in [PROTOTYPE_METHOD, "40-shot linear SVM"]:
        fp32 = prediction_arrays[(model_name, "FP32", method)]
        w8a8 = prediction_arrays[
            (model_name, "W8A8 dynamic PTQ", method)
        ]
        agreement_rows.append({
            "model": model_name,
            "precision": f"FP32 vs W8A8: {method}",
            "prototype_svm_agreement": float(np.mean(fp32 == w8a8)),
        })

camera_agreement = pl.DataFrame(agreement_rows)
display(
    camera_agreement.with_columns(
        pl.col("prototype_svm_agreement").round(3)
    )
)

# Compare FP32 and W8A8 camera embeddings independently of predictions.
camera_embedding_agreement = pl.DataFrame([
    {
        "model": model_name,
        "mean_fp32_w8a8_cosine": float(np.mean(np.sum(
            camera_fp32_bundles[model_name].features
            * camera_w8a8_bundles[model_name].features,
            axis=1,
        ))),
    }
    for model_name in CAMERA_MODEL_NAMES
])
display(camera_embedding_agreement.with_columns(
    pl.col("mean_fp32_w8a8_cosine").round(4)
))


In [ ]:
# Ask how each model variant distributes its choices among species.
prediction_counts = (
    camera_predictions.with_columns(
        (
            pl.col("model")
            + " | "
            + pl.col("precision")
            + " | "
            + pl.col("method")
        ).alias("variant")
    )
    .group_by(["predicted_species", "variant"])
    .len(name="images")
    .pivot(
        on="variant",
        index="predicted_species",
        values="images",
    )
    .fill_null(0)
    .sort("predicted_species")
)
with pl.Config(tbl_cols=-1, tbl_width_chars=220):
    display(prediction_counts)


## Inspect the strongest disagreements

The gallery selects images with the largest number of distinct predictions across the three model generations plus the adapted model, two precision settings, and two classifiers. This prioritizes unresolved cases rather than claiming that disagreement itself identifies an error.


In [ ]:
# Prioritize camera images that produce the widest range of predictions.
predictions_by_uuid = {}
for row in camera_predictions.iter_rows(named=True):
    predictions_by_uuid.setdefault(row["uuid"], set()).add(
        row["predicted_species"]
    )

disagreement_order = sorted(
    range(len(lila_uuids)),
    key=lambda index: (
        -len(predictions_by_uuid[str(lila_uuids[index])]),
        str(lila_uuids[index]),
    ),
)
inspection_indices = disagreement_order[:12]
inspection_uuids = [str(lila_uuids[index]) for index in inspection_indices]

# Link the prioritized images to their complete prediction table.
disagreement_browser = ImageBrowser(
    images=[lila_image_bytes[index] for index in inspection_indices],
    details=[
        format_details([
            ("Source", lila_metadata["publisher"][index]),
            ("Image ID", str(lila_uuids[index])[:8]),
            (
                "Distinct predictions",
                len(predictions_by_uuid[str(lila_uuids[index])]),
            ),
        ])
        for index in inspection_indices
    ],
    boxes=[camera_boxes[index] for index in inspection_indices],
    title="Camera-trap images prioritized by prediction disagreement",
    random_seed=RANDOM_SEED,
)
disagreement_browser.display()

inspection_predictions = (
    camera_predictions.filter(pl.col("uuid").is_in(inspection_uuids))
    .with_columns(pl.col("uuid").str.slice(0, 8).alias("image_id"))
    .select(
        "image_id",
        "publisher",
        "model",
        "precision",
        "method",
        "predicted_species",
        pl.col("margin").round(3),
    )
    .sort(["image_id", "model", "precision", "method"])
)
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=180):
    display(inspection_predictions)

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

## Discuss

For several images, inspect the pixels and the complete prediction table.

1. Is an animal visible, and is enough of it visible for species identification?
2. Which visible traits support or conflict with the predictions?
3. Does agreement across models and methods change how much you trust a prediction?
4. Would site, range, season, or neighboring frames change the plausible species?
5. What additional observation would resolve the identification?
6. How should a field pipeline handle empty frames, partial animals, and cases where every closed-set choice is doubtful?


## Reading the results

The evaluation boundary is an embedding bundle: ordered image UUIDs, normalized image embeddings, and a producer manifest. Zero-shot scoring and every few-shot SVM consume that artifact rather than a live model. Each SVM is refit for its bundle and support set; a future abstention threshold must be fit the same way.

The current W8A8 rows are TorchAO eager numerical experiments. They do not claim ExecuTorch or Raspberry Pi latency. A later exported or deployed producer can return a bundle with the same UUIDs and append results to these analyses. Its manifest must record the backend and delegated-versus-fallback operator coverage. Target-runtime latency remains blank until it is measured in that runtime.

Use the tables and figures together:

- Zero-shot results measure how well image and taxonomic text embeddings align without fitting on labeled task images.
- SVM and learning-curve results measure how useful each frozen representation is once labeled examples are available.
- The adaptation result measures whether targeted supervision of the final visual block improves this task while retaining the same embedding interface.
- Confusion matrices and error galleries show which species distinctions account for the aggregate score and which images merit inspection.
- t-SNE shows local structure worth investigating, but it is not a performance metric.
- Prediction flips show where quantization changes a task decision, even when the two embedding vectors remain very similar.
- Quantization should be judged across storage, measured latency, embedding agreement, and task performance. No single column settles the choice.

Limits of this exercise:

- The labeled-task UUIDs, split, and support sets are shared across models, but those images were present during BioCLIP 2 and BioCLIP 2.5 foundation training.
- The separate LILA cohort is marked unseen and has no species labels. Its predictions are hypotheses for inspection, not evaluation results.
- The random split does not control for site, specimen, photographer, or individual. Field evaluation may require grouped splits.
- Every classifier is closed-set. Unknown species and unsuitable images require an abstention or review policy.
- Timing depends on the accelerator, processor, thread count, batch size, and available kernels. Repeat the benchmark in the deployment runtime.


# 10/10 Check a broader set of tasks.

## BioBench: task breadth after quantization

[BioBench](https://github.com/Imageomics/biobench) evaluates frozen visual representations on application-driven biological tasks. Its standard probes use image embeddings and labels; text prompts and the CLIP text tower are not part of the score.

Before comparing models, define the scientific decision the model must support: the unit of observation, the required prediction, the errors that matter most, and an adequate performance threshold. Also identify how the intended sensor, site, season, and taxa differ from the benchmark. BioBench can screen model options, but it does not replace validation on data from the intended study.

The complete BioBench suite contains millions of images and is intended for cluster-scale evaluation. This notebook runs all 164 binary tasks in the [NeWT](https://github.com/visipedia/newt) benchmark included in BioBench. A control directly below selects a stratified proportion of its 36,032 images. Five tasks, one from each task cluster, are highlighted to make the aggregate score easier to interpret:

| Cluster | Task |
|---|---|
| Appearance | Plant pathology: healthy or sick |
| Behavior | Copulation present |
| Context | Nest present |
| Counting | Multiple species present |
| Gestalt | Field-notes sketch present |

Every task retains its published train/test split. A proportion of `1.0` reproduces the complete NeWT evaluation and is comparable to the NeWT column in BioBench's published results. Smaller runs are useful for iteration but are not official benchmark scores.

Only BioCLIP 2.5 is used to create new image embeddings here. The earlier sections compare model generations; this section asks whether W8A8 preserves the selected model's usefulness across a much broader set of tasks.


## Choose the NeWT run size

This setting controls both embedding creation and evaluation. Sampling is deterministic and stratified by task, train/test split, and class, so every task remains usable.

- `0.10` processes about 3,600 images and is the default quick run.
- `1.0` processes all 36,032 images and reproduces the full NeWT evaluation.

The image archive is distributed as one file, so this reduces inference and probe time but not the initial download.


In [ ]:
# Choose how much of NeWT to embed while retaining every task and split.
NEWT_SAMPLE_PROPORTION = 0.10
NEWT_SAMPLE_SEED = 42

if not 0 < NEWT_SAMPLE_PROPORTION <= 1:
    raise ValueError("NEWT_SAMPLE_PROPORTION must be greater than 0 and at most 1")

print(
    f"NeWT run: {NEWT_SAMPLE_PROPORTION:.0%} of images, "
    f"stratified with seed {NEWT_SAMPLE_SEED}"
)


## Download NeWT

The official NeWT image archive is about 4.1 GB. Extraction temporarily requires space for both the archive and its contents; the archive is deleted after a successful extraction.

In [ ]:
# Download and validate the NeWT labels and image archive when needed.
import os
from pathlib import Path

from biobench_helpers import (
    encode_image_paths,
    load_embedding_cache,
    prepare_newt,
    sample_newt_metadata,
)

NEWT_IMAGES_URL = (
    "https://ml-inat-competition-datasets.s3.amazonaws.com/"
    "newt/newt2021_images.tar.gz"
)
NEWT_LABELS_URL = (
    "https://ml-inat-competition-datasets.s3.amazonaws.com/"
    "newt/newt2021_labels.csv.tar.gz"
)
BIOBENCH_NEWT_DIR = Path(
    os.environ.get("BIOBENCH_NEWT_DIR", "biobench_data/newt")
).expanduser()
EXPECTED_NEWT_IMAGES = 36_032
NEWT_IMAGES_DIR, NEWT_LABELS_PATH = prepare_newt(
    BIOBENCH_NEWT_DIR,
    images_url=NEWT_IMAGES_URL,
    labels_url=NEWT_LABELS_URL,
    expected_images=EXPECTED_NEWT_IMAGES,
)

print(f"NeWT directory: {BIOBENCH_NEWT_DIR.resolve()}")
print(f"Labels available: {NEWT_LABELS_PATH.exists()}")
print(f"Images available: {EXPECTED_NEWT_IMAGES:,}")


## Prepare all 164 tasks

The requested proportion is sampled independently within each task, published split, and binary class. At least one row is retained from every stratum. We calculate accuracy and macro-F1 for each task so changes hidden by the aggregate can be inspected.


In [ ]:
# Sample every NeWT task and identify five tasks for closer inspection.
BIOBENCH_TASKS = {
    "fgvcx_plant_pathology_healthy_vs_sick": {
        "cluster": "appearance",
        "display_name": "Plant health",
    },
    "ml_tag_copulation": {
        "cluster": "behavior",
        "display_name": "Copulation",
    },
    "ml_tag_nest": {
        "cluster": "context",
        "display_name": "Nest",
    },
    "ml_tag_multiple_species": {
        "cluster": "counting",
        "display_name": "Multiple species",
    },
    "ml_tag_field_notes_sketch": {
        "cluster": "gestalt",
        "display_name": "Field-notes sketch",
    },
}

newt_metadata, newt_full_metadata, newt_paths = sample_newt_metadata(
    NEWT_LABELS_PATH,
    NEWT_IMAGES_DIR,
    proportion=NEWT_SAMPLE_PROPORTION,
    seed=NEWT_SAMPLE_SEED,
    display_names={
        task: details["display_name"]
        for task, details in BIOBENCH_TASKS.items()
    },
    expected_rows=EXPECTED_NEWT_IMAGES,
)

# Check the realized sample size and the highlighted task balance.
suite_summary = newt_metadata.select(
    pl.lit(NEWT_SAMPLE_PROPORTION).alias("requested_proportion"),
    (pl.len() / newt_full_metadata.height).alias("realized_proportion"),
    pl.len().alias("images"),
    pl.col("id").n_unique().alias("unique_images"),
    pl.col("task").n_unique().alias("tasks"),
    (pl.col("split") == "train").sum().alias("train_images"),
    (pl.col("split") == "test").sum().alias("test_images"),
)
display(suite_summary)

highlight_balance = (
    newt_metadata.filter(pl.col("task").is_in(list(BIOBENCH_TASKS)))
    .group_by(["cluster", "display_name", "split", "label"])
    .len(name="images")
    .sort(["cluster", "display_name", "split", "label"])
)
display(highlight_balance)


In [ ]:
# Browse one source image from each highlighted scientific signal.
highlight_rows = [
    newt_metadata.filter(pl.col("task") == task_name)
    .sort(["label", "id"])
    .row(0, named=True)
    for task_name in BIOBENCH_TASKS
]
newt_browser = ImageBrowser(
    images=[
        NEWT_IMAGES_DIR / f"{row['id']}.jpg" for row in highlight_rows
    ],
    details=[
        format_details([
            ("Cluster", row["cluster"]),
            ("Task", row["display_name"]),
            ("Label", row["text_label"]),
        ])
        for row in highlight_rows
    ],
    title="Highlighted NeWT tasks",
    random_seed=RANDOM_SEED,
)
newt_browser.display()

# You may need to do a hard refresh of this page for the widget to render. If you do so, rerun this cell as well.

## Create BioCLIP 2.5 image embeddings before and after W8A8

FP32 and W8A8 process the same selected image files with the same evaluation transform. Embeddings are cached separately by model revision, quantization implementation, device type, and the exact selected image IDs. A first run performs both image passes; later runs reuse the matching caches.

The text tower is present in the CLIP checkpoint but is not used for this evaluation. Any score change comes from the visual representation and the probe trained on top of it.


In [ ]:
# Create matched FP32 and W8A8 BioCLIP 2.5 embeddings for NeWT.
BIOBENCH_MODEL_NAME = "BioCLIP 2.5"
BIOBENCH_MODEL_SPEC = MODEL_SPECS[BIOBENCH_MODEL_NAME]
BIOBENCH_BATCH_SIZE = 32 if QUANT_DEVICE.type == "cuda" else 8
BIOBENCH_WORKERS = min(4, os.cpu_count() or 1)
BIOBENCH_SUITE_ID = "newt-164-tasks-v1"
BIOBENCH_CACHE_DIR = Path("biobench_embedding_cache")
BIOBENCH_CACHE_DIR.mkdir(exist_ok=True)
biobench_ids = np.asarray(newt_metadata["id"].to_list(), dtype=np.str_)
BIOBENCH_SELECTION_ID = ids_digest(biobench_ids)[:12]

# Tie each cache name to the selected rows, model revision, and device type.
if NEWT_SAMPLE_PROPORTION == 1:
    # Reuse complete-run caches from earlier notebook versions.
    cache_prefix = (
        f"{BIOBENCH_SUITE_ID}_{BIOBENCH_MODEL_SPEC['revision'][:8]}_"
        f"{QUANT_DEVICE.type}"
    )
else:
    cache_prefix = (
        f"{BIOBENCH_SUITE_ID}_{BIOBENCH_SELECTION_ID}_"
        f"{BIOBENCH_MODEL_SPEC['revision'][:8]}_{QUANT_DEVICE.type}"
    )
fp32_cache_path = BIOBENCH_CACHE_DIR / f"{cache_prefix}_fp32.npz"
w8a8_cache_path = BIOBENCH_CACHE_DIR / (
    f"{cache_prefix}_{QUANTIZATION_CACHE_TAG}.npz"
)
# Describe the two embedding producers before loading either cache.
biobench_fp32_manifest = producer_manifest(
    bundle_id=(
        f"{BIOBENCH_SUITE_ID}:{BIOBENCH_SELECTION_ID}:"
        f"{BIOBENCH_MODEL_NAME}:fp32"
    ),
    model_name=BIOBENCH_MODEL_NAME,
    repo_id=BIOBENCH_MODEL_SPEC["repo_id"],
    revision=BIOBENCH_MODEL_SPEC["revision"],
    precision="FP32",
    evidence_type="numerical experiment",
    framework="OpenCLIP",
    framework_version=open_clip.__version__,
    preprocessing="OpenCLIP evaluation transform from pinned model config",
    quantization=None,
    dataset={
        "name": "NeWT",
        "suite_id": BIOBENCH_SUITE_ID,
        "sample_proportion": NEWT_SAMPLE_PROPORTION,
        "sample_seed": NEWT_SAMPLE_SEED,
        "selection_id": BIOBENCH_SELECTION_ID,
    },
    backend="Torch eager",
    runtime={"device_type": QUANT_DEVICE.type, "target_runtime_latency_comparable": False},
)
biobench_w8a8_manifest = w8a8_producer_manifest(
    bundle_id=(
        f"{BIOBENCH_SUITE_ID}:{BIOBENCH_SELECTION_ID}:"
        f"{BIOBENCH_MODEL_NAME}:w8a8-dynamic"
    ),
    model_name=BIOBENCH_MODEL_NAME,
    repo_id=BIOBENCH_MODEL_SPEC["repo_id"],
    revision=BIOBENCH_MODEL_SPEC["revision"],
    dataset={
        "name": "NeWT",
        "suite_id": BIOBENCH_SUITE_ID,
        "sample_proportion": NEWT_SAMPLE_PROPORTION,
        "sample_seed": NEWT_SAMPLE_SEED,
        "selection_id": BIOBENCH_SELECTION_ID,
    },
    device_type=QUANT_DEVICE.type,
)
biobench_fp32_bundle = load_embedding_cache(
    fp32_cache_path, biobench_ids, biobench_fp32_manifest
)
biobench_w8a8_bundle = load_embedding_cache(
    w8a8_cache_path, biobench_ids, biobench_w8a8_manifest
)

# Compute only the missing representation, then save its validated bundle.
if biobench_fp32_bundle is None or biobench_w8a8_bundle is None:
    snapshot = snapshot_download(
        repo_id=BIOBENCH_MODEL_SPEC["repo_id"],
        revision=BIOBENCH_MODEL_SPEC["revision"],
        allow_patterns=[
            "open_clip_config.json",
            BIOBENCH_MODEL_SPEC["weight_file"],
        ],
    )
    model, _, preprocess = open_clip.create_model_and_transforms(
        f"local-dir:{snapshot}",
        device=QUANT_DEVICE,
        precision="fp32",
    )
    model.eval()

    if biobench_fp32_bundle is None:
        print(f"Creating FP32 image embeddings with {BIOBENCH_MODEL_NAME}...")
        biobench_fp32_features = encode_image_paths(
            model,
            preprocess,
            newt_paths,
            batch_size=BIOBENCH_BATCH_SIZE,
            workers=BIOBENCH_WORKERS,
            device=QUANT_DEVICE,
        )
        biobench_fp32_bundle = EmbeddingBundle.create(
            biobench_ids, biobench_fp32_features, biobench_fp32_manifest
        )
        biobench_fp32_bundle.save(fp32_cache_path)
    else:
        print(f"Loaded FP32 embeddings from {fp32_cache_path}")

    if biobench_w8a8_bundle is None:
        print(f"Quantizing {BIOBENCH_MODEL_NAME}...")
        quantize_(model.visual, make_w8a8_config())
        model.eval()
        print(f"Creating W8A8 image embeddings with {BIOBENCH_MODEL_NAME}...")
        biobench_w8a8_features = encode_image_paths(
            model,
            preprocess,
            newt_paths,
            batch_size=BIOBENCH_BATCH_SIZE,
            workers=BIOBENCH_WORKERS,
            device=QUANT_DEVICE,
        )
        biobench_w8a8_bundle = EmbeddingBundle.create(
            biobench_ids, biobench_w8a8_features, biobench_w8a8_manifest
        )
        biobench_w8a8_bundle.save(w8a8_cache_path)
    else:
        print(f"Loaded W8A8 embeddings from {w8a8_cache_path}")

    del model, preprocess
    gc.collect()
    release_device_memory(QUANT_DEVICE)
else:
    print(f"Loaded FP32 embeddings from {fp32_cache_path}")
    print(f"Loaded W8A8 embeddings from {w8a8_cache_path}")

# Validate both feature arrays before fitting any task probes.
biobench_fp32_features = biobench_fp32_bundle.features
biobench_w8a8_features = biobench_w8a8_bundle.features

for precision, features in {
    "FP32": biobench_fp32_features,
    "W8A8": biobench_w8a8_features,
}.items():
    norms = np.linalg.norm(features, axis=1)
    if not np.isfinite(features).all() or np.any(norms == 0):
        raise ValueError(f"Invalid {precision} NeWT embeddings")

print("FP32:", biobench_fp32_features.shape)
print("W8A8:", biobench_w8a8_features.shape)


## Evaluate all 164 tasks

This follows BioBench's NeWT evaluation procedure on the selected rows. For each task, the training mean is subtracted from both splits, each resulting embedding is L2-normalized, and an independent linear SVM is fit using the retained training examples. FP32 and W8A8 use identical rows and probes.

The complete-run NeWT score is accuracy across all test images. Equal-task mean accuracy and macro-F1 are also shown because they prevent larger tasks from dominating the summary. With a proportion below `1.0`, all values describe the deterministic sample rather than an official NeWT score.


In [ ]:
# Fit the same centered linear probe for every task and representation.
biobench_feature_sets = {
    "FP32": biobench_fp32_features,
    "W8A8 dynamic PTQ": biobench_w8a8_features,
}
biobench_metric_rows = []

task_values = newt_metadata["task"].to_numpy()
split_values = newt_metadata["split"].to_numpy()
label_values = newt_metadata["label"].to_numpy()


def center_and_normalize(train_features, test_features):
    train_mean = train_features.mean(axis=0, keepdims=True)
    train_features = train_features - train_mean
    test_features = test_features - train_mean
    train_features /= np.linalg.norm(
        train_features, axis=1, keepdims=True
    ).clip(min=1e-12)
    test_features /= np.linalg.norm(
        test_features, axis=1, keepdims=True
    ).clip(min=1e-12)
    return train_features, test_features


# Preserve each task's published train/test split while fitting its probe.
for precision, features in biobench_feature_sets.items():
    for task_name in newt_metadata["task"].unique().sort():
        train_mask = (task_values == task_name) & (split_values == "train")
        test_mask = (task_values == task_name) & (split_values == "test")

        x_train, x_test = center_and_normalize(
            features[train_mask].copy(), features[test_mask].copy()
        )
        probe = SVC(kernel="linear")
        probe.fit(x_train, label_values[train_mask])
        predictions = probe.predict(x_test)
        truth = label_values[test_mask]
        task_row = newt_metadata.filter(
            pl.col("task") == task_name
        ).row(0, named=True)
        display_name = BIOBENCH_TASKS.get(task_name, {}).get(
            "display_name", task_name
        )
        biobench_metric_rows.append({
            "cluster": task_row["task_cluster"],
            "task_id": task_name,
            "task": display_name,
            "precision": precision,
            "test_images": int(test_mask.sum()),
            "test_correct": int((truth == predictions).sum()),
            "accuracy": accuracy_score(truth, predictions),
            "macro_f1": f1_score(truth, predictions, average="macro"),
            "highlighted": task_name in BIOBENCH_TASKS,
        })

biobench_metrics = pl.DataFrame(biobench_metric_rows).sort(
    ["cluster", "task_id", "precision"]
)

# Ask how the two representations compare across the full selected suite.
biobench_suite_summary = (
    biobench_metrics.group_by("precision")
    .agg(
        (pl.col("test_correct").sum() / pl.col("test_images").sum())
        .alias("newt_accuracy"),
        pl.col("accuracy").mean().alias("mean_task_accuracy"),
        pl.col("macro_f1").mean().alias("mean_task_macro_f1"),
        pl.len().alias("tasks"),
        pl.col("test_images").sum().alias("test_images"),
    )
    .sort("precision")
)
display(
    biobench_suite_summary.with_columns(
        pl.col(["newt_accuracy", "mean_task_accuracy", "mean_task_macro_f1"])
        .round(4)
    )
)

# Isolate the five highlighted tasks for a concrete paired comparison.
highlighted_metrics = biobench_metrics.filter(pl.col("highlighted"))
display(
    highlighted_metrics.select(
        "cluster", "task", "precision", "test_images", "accuracy", "macro_f1"
    ).with_columns(pl.col(["accuracy", "macro_f1"]).round(3))
)

fp32_task_scores = (
    highlighted_metrics.filter(pl.col("precision") == "FP32")
    .select(
        "cluster",
        "task",
        pl.col("macro_f1").alias("fp32_macro_f1"),
    )
)
w8a8_task_scores = (
    highlighted_metrics.filter(pl.col("precision") == "W8A8 dynamic PTQ")
    .select(
        "cluster",
        "task",
        pl.col("macro_f1").alias("w8a8_macro_f1"),
    )
)
biobench_task_comparison = (
    fp32_task_scores.join(
        w8a8_task_scores, on=["cluster", "task"], how="inner"
    )
    .with_columns(
        (
            pl.col("w8a8_macro_f1") - pl.col("fp32_macro_f1")
        ).alias("w8a8_minus_fp32")
    )
    .sort(["cluster", "task"])
)
display(
    biobench_task_comparison.with_columns(
        pl.exclude(["cluster", "task"]).round(3)
    )
)

In [ ]:
# Compare embedding agreement and macro-F1 on the highlighted tasks.
cosine_agreement = np.sum(
    biobench_fp32_features * biobench_w8a8_features,
    axis=1,
)
print(
    "Embedding agreement: "
    f"mean={cosine_agreement.mean():.4f}, "
    f"p05={np.quantile(cosine_agreement, 0.05):.4f}, "
    f"minimum={cosine_agreement.min():.4f}"
)

comparison_plot = biobench_task_comparison.sort("cluster")
plot_labels = [
    f"{cluster}\n{task}"
    for cluster, task in comparison_plot.select(
        "cluster", "task"
    ).iter_rows()
]
x = np.arange(len(plot_labels))
width = 0.36

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(
    x - width / 2,
    comparison_plot["fp32_macro_f1"],
    width,
    label="FP32",
)
ax.bar(
    x + width / 2,
    comparison_plot["w8a8_macro_f1"],
    width,
    label="W8A8 dynamic PTQ",
)
ax.set(
    ylabel="Macro F1",
    title="Highlighted NeWT tasks before and after W8A8",
    xticks=x,
    xticklabels=plot_labels,
    ylim=(0, 1),
)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## Compare all 164 tasks

Each point is one NeWT task. Points above the diagonal improved after quantization; points below it declined. The histogram shows the paired accuracy changes directly, while labels connect the five highlighted tasks to their image examples.

In [ ]:
# Ask how many individual tasks improve, tie, or decline after W8A8.
newt_task_changes = (
    biobench_metrics.select(
        "cluster", "task_id", "task", "highlighted",
        "precision", "test_images", "accuracy",
    )
    .pivot(
        on="precision",
        index=["cluster", "task_id", "task", "highlighted", "test_images"],
        values="accuracy",
    )
    .rename({
        "FP32": "fp32_accuracy",
        "W8A8 dynamic PTQ": "w8a8_accuracy",
    })
    .with_columns(
        (pl.col("w8a8_accuracy") - pl.col("fp32_accuracy"))
        .alias("w8a8_minus_fp32")
    )
    .sort(["cluster", "task_id"])
)
newt_change_counts = newt_task_changes.select(
    (pl.col("w8a8_minus_fp32") > 0).sum().alias("improved_tasks"),
    (pl.col("w8a8_minus_fp32") == 0).sum().alias("tied_tasks"),
    (pl.col("w8a8_minus_fp32") < 0).sum().alias("declined_tasks"),
)
display(newt_change_counts)

# Show every paired task score and the distribution of score changes.
cluster_colors = {
    "appearance": "#1b9e77",
    "behavior": "#d95f02",
    "context": "#7570b3",
    "counting": "#e7298a",
    "gestalt": "#66a61e",
}
fig, (scatter_ax, histogram_ax) = plt.subplots(
    1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.2, 1]}
)
for cluster, color in cluster_colors.items():
    cluster_rows = newt_task_changes.filter(pl.col("cluster") == cluster)
    scatter_ax.scatter(
        cluster_rows["fp32_accuracy"],
        cluster_rows["w8a8_accuracy"],
        s=42, alpha=0.72, color=color, label=cluster,
    )

plot_min = float(min(
    newt_task_changes["fp32_accuracy"].min(),
    newt_task_changes["w8a8_accuracy"].min(),
)) - 0.02
scatter_ax.plot([plot_min, 1], [plot_min, 1], color="#444444", linestyle="--")
for row in newt_task_changes.filter(pl.col("highlighted")).iter_rows(named=True):
    scatter_ax.annotate(
        row["task"],
        (row["fp32_accuracy"], row["w8a8_accuracy"]),
        xytext=(5, 5), textcoords="offset points", fontsize=8,
    )
scatter_ax.set(
    xlim=(plot_min, 1.01), ylim=(plot_min, 1.01),
    xlabel="FP32 task accuracy", ylabel="W8A8 task accuracy",
    title="Paired NeWT task performance",
)
scatter_ax.set_aspect("equal", adjustable="box")
scatter_ax.legend(title="Task cluster", fontsize=9)
scatter_ax.grid(alpha=0.2)

deltas = newt_task_changes["w8a8_minus_fp32"].to_numpy()
histogram_ax.hist(deltas, bins=25, color="#4c78a8", edgecolor="white")
histogram_ax.axvline(0, color="#444444", linestyle="--")
histogram_ax.axvline(
    deltas.mean(), color="#b2182b", linewidth=2,
    label=f"mean {deltas.mean():+.3f}",
)
histogram_ax.set(
    xlabel="W8A8 minus FP32 task accuracy",
    ylabel="Tasks", title="Distribution of task changes",
)
histogram_ax.legend()
histogram_ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## Published BioBench context

BioBench publishes machine-readable full-data NeWT results for many frozen backbones. The table below retrieves a small set of relevant reference models and places this run beside them. Published confidence intervals come from BioBench's bootstrap analysis.

The local result is directly comparable only when `NEWT_SAMPLE_PROPORTION = 1.0`. A sampled result is labeled with its realized proportion and has no confidence interval.


In [ ]:
# Place this run beside selected published full-data NeWT results.
BIOBENCH_RESULTS_URL = (
    "https://raw.githubusercontent.com/Imageomics/biobench/"
    "main/docs/data/results.json"
)
reference_model_ids = [
    "ViT-B-16/openai",
    "ViT-L-14/openai",
    "ViT-SO400M-14-SigLIP-384/webli",
    "dinov2_vitl14_reg",
    "hf-hub:imageomics/bioclip",
    "hf-hub:imageomics/bioclip-2",
    "hf-hub:imageomics/bioclip-2.5-vith14",
]

response = requests.get(BIOBENCH_RESULTS_URL, timeout=60)
response.raise_for_status()
published = response.json()
model_display = {
    row["ckpt"]: row["display"] for row in published["models"]
}
newt_results = {
    row["model"]: row
    for row in published["results"]
    if row["task"] == "newt"
}

# Retain source labels so sampled and published scores are not conflated.
reference_rows = []
for model_id in reference_model_ids:
    result = newt_results[model_id]
    reference_rows.append({
        "source": "Published BioBench",
        "model": model_display[model_id],
        "newt_accuracy": result["mean"],
        "ci_low": result["ci_low"],
        "ci_high": result["ci_high"],
    })

realized_proportion = float(suite_summary["realized_proportion"][0])
run_source = (
    "This run"
    if NEWT_SAMPLE_PROPORTION == 1
    else f"This run ({realized_proportion:.1%} stratified sample)"
)
for row in biobench_suite_summary.iter_rows(named=True):
    reference_rows.append({
        "source": run_source,
        "model": f"BioCLIP 2.5 {row['precision']}",
        "newt_accuracy": row["newt_accuracy"],
        "ci_low": None,
        "ci_high": None,
    })

biobench_reference_table = pl.DataFrame(reference_rows)
biobench_reference_table.with_columns(
    pl.col(["newt_accuracy", "ci_low", "ci_high"]).round(4)
)


## Interpreting the results

This section measures all NeWT tasks, but still covers only one component of BioBench:

- With a full-data run, agreement between the local FP32 score and the published BioCLIP 2.5 score checks the evaluation before quantization is interpreted.
- A sampled run is a faster check of the pipeline and FP32-to-W8A8 direction, not a replacement for the published score.
- A small W8A8-minus-FP32 difference across the 164 tasks suggests that quantization preserved broadly useful visual information.
- The highlighted tasks show concrete examples of changes that the aggregate can hide.
- Similar aggregate scores can still contain opposing task-level changes; the full `biobench_metrics` table is available for inspection.
- NeWT broadens the prediction questions beyond *Peromyscus*, but it does not reproduce BioBench's full range of kingdoms and acquisition modalities.
